<div align="center">

# Hafta 8: Çok modlu yapay zekâ ve veri füzyonu

**Python ile Yapay Zekâ Eğitimi: Kardiyologlar için Uygulamalı Eğitim Programı**<br>
Türk Kardiyoloji Derneği · Prof. Dr. Utku Köse

[Hafta sayfası](README.md) · [Ders sayfası](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-08/ders.html) · [Etkileşimli lab](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-08/lab.html) · [PDF ders notu](Hafta08_Ders_Notlari.pdf) · [Colab'da aç](https://colab.research.google.com/github/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/tr/hafta-08/NB08_cok_modlu_yapay_zeka_ve_veri_fuzyonu.ipynb)

</div>

**Bu Jupyter Notebook iki kısımdan oluşur.** Birincisinde kodu bir yapay zekâ (YZ) asistanı yazar; asistana ne yapacağını yazılı bir talimat, yani bir prompt (istem) anlatır. İkincisinde kodu Python ile kendiniz yazarsınız. İki kısım da aynı verilerle aynı sonuçlara varır; ders sayfasındaki her uygulama durağında bunlardan birini seçin.

**Ders sayfasıyla birlikte.** Bu Jupyter Notebook'a ders sayfasının uygulama duraklarından gelinir. Her durağın sonundaki bağlantı sizi ders sayfasına, okumanın devam ettiği yere geri götürür.

| | Kimler için? | Ne yapılır? |
|---|---|---|
| **[Kısım 1: Prompt yolu](#part-1)** | Kod yazmayan katılımcılar | Kodu, hazır promptlarla bir YZ asistanı yazar; siz çalıştırır ve sonucu denetlersiniz. |
| **[Kısım 2: Python yolu](#part-2)** | Python öğrenmek isteyen katılımcılar | Kodu alıştırmalarla adım adım kendiniz yazar ve çalıştırırsınız. |

**Nerede çalışır?** Aynı Jupyter Notebook, hiçbir kurulum gerektirmeyen Google Colab'da da kendi bilgisayarınızdaki Jupyter'de de (Anaconda; ders klasöründeki SETUP.tr.md dosyasına bakın) çalışır. Hangisini tercih ederseniz onu kullanın. Colab'da önce *Dosya → Drive'a kopya kaydet* (*File → Save a copy in Drive*) komutuyla kendi kopyanızı kaydedin; aksi hâlde sekme kapanınca yaptığınız değişiklikler kaybolur. 6. bölümdeki EKG ağı ve 7. bölümdeki iki füzyon ağının her biri sıradan bir dizüstü bilgisayarda bir ile üç dakikada eğitilir; ekran kartı gerekmez. Hangi kısmı seçerseniz seçin önce 0. bölümü çalıştırın. Bu bölüm ders dosyalarını ve paketleri hazırlar.

**Veri ve mahremiyet.** Buradaki bütün veri setleri ya sentetiktir ya da açık bir araştırma veri tabanından gelmektedir. Gerçek hasta verisi yalnızca kendi bilgisayarınızda, kurumunuzun kurallarına uygun olarak ve etik kurul onayıyla işlenir; bulut tabanlı bir Jupyter Notebook'a ya da bir YZ sohbetine asla konmaz.

## 0. Kurulum

İlk hücre ders klasörünü bulur. Jupyter Notebook tek başına açıldıysa, örneğin Google Colab'da, `vclinic` ders paketini ve veri dosyalarını GitHub'dan indirir. İkinci hücre paketleri yükler, mesajların dilini seçer ve haftanın isteğe bağlı paketlerini denetler. Hiçbir şey indirilmez: Kayıt sistemi, A ve B hastanelerinin göğüs ağrısı ünitesi ve EKG'leri ders paketi tarafından bu bilgisayarda bir dakikadan kısa sürede çizilir. 

In [ ]:
#@title Ders dosyaları: Bu hücreyi çalıştırın (açmanıza gerek yok)
import pathlib
import sys
import urllib.request

REPO_RAW = "https://raw.githubusercontent.com/utkukose/ai-cardiology-tkd-NB-lecture/main/"
DATA_FILES = [
]


def find_course_root():
    here = pathlib.Path.cwd().resolve()
    for folder in [here, *here.parents]:
        if (folder / "vclinic" / "__init__.py").exists():
            return folder
    return None


ROOT = find_course_root()
if ROOT is None or ROOT.name == "ai-cardiology-course":
    # İndirilmiş bir kopyada, örneğin Colab'da, paketin dosyaları yenilenir.
    ROOT = pathlib.Path.cwd() / "ai-cardiology-course" if ROOT is None else ROOT
    manifest = urllib.request.urlopen(REPO_RAW + "vclinic/MANIFEST.txt").read().decode("utf-8")
    package = ["vclinic/" + line.strip() for line in manifest.splitlines() if line.strip()]
    for rel in package + DATA_FILES:
        target = ROOT / rel
        if rel in package or not target.exists():
            target.parent.mkdir(parents=True, exist_ok=True)
            urllib.request.urlretrieve(REPO_RAW + rel, target)
    print("Ders dosyaları şu klasöre indirildi:", ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [ ]:
# Kurulum: paketler, dil ve çizim biçimi
import importlib

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import copy

from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss, roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from vclinic import cohort, followup, triage

try:
    import torch
    import torch.nn as nn
except ImportError as error:
    raise ImportError("PyTorch kurulu değil. SETUP.tr.md dosyasında anlatıldığı gibi kurun, ardından çekirdeği yeniden başlatın.") from error
import vclinic
from vclinic import teach

if tuple(int(x) for x in vclinic.__version__.split(".")) < (1, 9, 0):
    raise ImportError("vclinic ders paketi bu Jupyter Notebook'tan daha eskidir. Ders klasörünü güncelleyin (git pull komutuyla ya da ZIP dosyasını yeniden indirerek) ve çekirdeği yeniden başlatın.")
teach.set_language("tr")
plt.rcParams.update({"figure.dpi": 110, "font.family": "DejaVu Sans"})
RANDOM_STATE = 2026
torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.set_num_threads(min(4, torch.get_num_threads()))
print("PyTorch", torch.__version__)
print("vclinic sürümü", vclinic.__version__)
print("Ders klasörü:", ROOT)
for name in []:
    try:
        importlib.import_module(name)
        print(f"{name}: kurulu")
    except ImportError:
        print(f"{name}: kurulu değil (isteğe bağlı; Jupyter Notebook yerleşik bir yedek yol kullanır)")

In [ ]:
#@title Alıştırmaların beklenen cevapları (bu hücreyi çalıştırın; açmayı alıştırmaları denedikten sonraya bırakın)
teach.register_answers({
    "Alıştırma 8.1": (0.675, 0.001),
    "Alıştırma 8.2": (0.26, 0.001),
    "Alıştırma 8.3": (1378, 0),
    "Alıştırma A": (34, 0),
    "Alıştırma B": (0.837, 0.002),
    "Alıştırma C": (0.998, 0.01),
    "Alıştırma D": (0.021, 0.001),
    "Alıştırma E": (304, 0),
    "Alıştırma F": (75105, 0),
    "Alıştırma G": (20833, 0),
    "Alıştırma H": (0.903, 0.01),
    "Alıştırma I": (0.105, 0.005),
    "Alıştırma J": (0.139, 0.01),
    "Adım 1.1": (0.849, 0.002),
    "Adım 1.2": (0.83, 0.002),
    "Adım 1.3": (0.833, 0.002),
    "Adım 1.4": (0.041, 0.03),
    "Adım 2.1": (0.903, 0.001),
    "Adım 2.2": (0.9, 0.002),
    "Adım 2.3": (0.922, 0.005),
    "Adım 2.4": (2.78, 0.1),
    "Adım 3.1": (0.37, 0.05),
    "Adım 3.2": (9, 2),
    "Adım 3.3": (0.125, 0.01),
})
print("Beklenen cevaplar hazır.")

In [ ]:
#@title Alıştırma promptlarının tabloları (bu hücreyi çalıştırın; Python yolunda okunması gerekmez)
def alistirma_tablosu(anahtar):
    """Bir alıştırma promptunun "veri" tablosu; haftanın verilerinden kurulur."""
    g = globals()
    k = anahtar
    need = lambda *names: [g[n] for n in names] if all(n in g for n in names) else (_ for _ in ()).throw(
        RuntimeError("Önce bu alıştırmadan önceki bölümleri çalıştırın."))
    if k in ("8.1", "8.2", "8.3"):
        u = triage.presentations("A")
        return pd.DataFrame({"başvuru": u["patient_id"], "bir saat sonra troponin": (u["troponin_1h"] / u["troponin_url"]).round(2),
                             "ekokardiyogram yapıldı": u["echo_done"], "MACE": u["mace30"]})
    if k == "A":
        registry, = need("registry")
        return pd.DataFrame({"hasta": registry["patient_id"], "küme": registry["set"].map({"training": "eğitim", "validation": "doğrulama", "test": "test"}),
                             "olay": registry["event24"]})
    if k in ("B", "C"):
        probs_hf, rte, yr = need("probs_hf", "rte", "yr")
        return pd.DataFrame({"olasılık, EKG": probs_hf["ECG"][rte], "olasılık, ekokardiyogram": probs_hf["eko"][rte],
                             "olasılık, laboratuvar": probs_hf["laboratuvar"][rte], "olay": yr[rte]})
    if k == "D":
        A, te = need("A", "te")
        return pd.DataFrame({"HEART": A["heart"].to_numpy()[te], "MACE": A["mace30"].to_numpy()[te]})
    if k == "H":
        XE, parts_A, early, tr, te, y = need("XE", "parts_A", "early", "tr", "te", "y")
        XE_ = XE.copy()
        cols = parts_A[4].columns
        XE_[cols] = XE.loc[tr, cols].mean().to_numpy()
        return pd.DataFrame({"not olmadan olasılık": early.predict_proba(XE_[te])[:, 1], "MACE": y[te]})
    if k == "I":
        pB_new, rest, yB = need("pB_new", "rest", "yB")
        return pd.DataFrame({"olasılık": pB_new[rest], "MACE": yB[rest]})
    if k == "J":
        P, te, y = need("P", "te", "y")
        return pd.DataFrame({"olasılık": P["early"][te], "MACE": y[te]})
    raise KeyError(k)
print("Alıştırma promptlarının tabloları hazır.")

<a id="part-1"></a>

# Kısım 1: Prompt yolu

**Kimler için?** Bu kısım kod yazmayan katılımcılar içindir. Burada hiç kod yazmazsınız. Kodu, hazır bir prompttan, yani yazılı bir talimattan yararlanarak bir yapay zekâ (YZ) asistanı yazar; siz kodu çalıştırır ve sonucu denetlersiniz.

**Neye ihtiyacınız var?** Python kodu yazabilen bir YZ asistanı yeterlidir. Google Colab'ın içinde Gemini bulunur; pencerenin üstündeki Gemini düğmesiyle açılır. Başka bir tarayıcı sekmesinde açılan ChatGPT, Claude, Copilot ya da Gemini de aynı işi görür.

**Bir adım nasıl ilerler?**

1. Adımın amacını okuyun.
2. Promptun altındaki veri hücresini çalıştırın. Hücre, promptun tarif ettiği "veri" tablosunu hazırlar ve tablonun ilk ve son satırlarını gösterir.
3. Promptu kopyalayın: Fareyle seçin ve Ctrl+C'ye (Mac'te Cmd+C) basın.
4. Promptu YZ asistanına yapıştırın ve gönderin.
5. Asistanın verdiği kodu kopyalayın, veri hücresinin altındaki hücreye `#` ile başlayan satırın yerine yapıştırın ve hücreyi ▶ düğmesiyle ya da Shift+Enter ile çalıştırın.
6. Hemen altındaki kontrol hücresini çalıştırın. Hücre *doğru ✓* ya da *henüz değil ✗* yazar.
7. *Henüz değil* yazarsa aşağıdaki onarım promptunu kodla birlikte asistana gönderin. Sonuç yine yanlışsa *Hazır kod* hücresini çalıştırın. Bu hücre her zaman doğru sonucu verir; böylece bir sonraki adıma geçebilirsiniz.

**"veri" ve "cevap".** 1. haftada olduğu gibi her adımın veri hücresi, promptun tarif ettiği "veri" tablosunu hazırlar. Asistan sonucu "cevap" adıyla saklar; kontrol hücresi de sonucu bu addan okur.

**Her durağın başında** o durağın hazırlık hücresini bir kez çalıştırın. Hücre durağın verilerini yükler; böylece bir durağa başka bir gün de başlanabilir.

**Onarım promptu** (çalışan ama kontrolden geçmeyen kod için):

> Aşağıdaki kod şu görev için yazıldı: [promptun Görev satırını yapıştırın]. Kod çalışıyor, ancak dersin kontrolü "henüz değil" yazıyor. Hatayı bul, iki cümleyle açıkla ve düzeltilmiş kodu ver. Sonuç yine "cevap" adıyla saklanmalı.  
>
> [kodu buraya yapıştırın]  

**Hata mesajı için prompt** (hata vererek duran kod için):

> Aşağıdaki kod bir Jupyter Notebook'ta çalıştırıldığında altta verilen hata mesajını veriyor. Hatanın nedenini sade bir dille açıkla ve düzeltilmiş kodu ver.  
>
> [kodu buraya yapıştırın]  
>
> [hata mesajını buraya yapıştırın]  

Haftanın bütün promptları [prompt kütüphanesinde](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/tr/hafta-08/PROMPTLAR.md) de yer alır; GitHub her promptun yanında bir kopyalama düğmesi gösterir.

**Hiç değişmeyen kural.** Promptlarda yalnızca dersin sentetik ya da açık verileri bulunur. Bir YZ asistanına gerçek hasta verisi asla yapıştırılmaz.

## Durak 1: EKG, ekokardiyogram ve laboratuvarla kalp yetersizliği olayları

*Ders: [1. uygulama durağı](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-08/ders.html#stop-1)* · *Python yolunda aynı sonuç: Kısım 2, 1-3. bölümler*

Bu durak oturumun mini projesidir. Sanal kliniğin kayıt sistemindeki hastalar üç modaliteyle tarif edilir: EKG, ekokardiyogram ve laboratuvar değerleri. Her modalite için tek başına ve üçü için birlikte bir model kurulur, erken ve geç füzyon karşılaştırılır; son adım ise füzyon ile en iyi tek modalite arasındaki farktan ne kadar emin olunabileceğini sorar.

**Durağın sonunda:** Tek başına laboratuvar değerlerinin (0,849), üç modalitenin erken füzyonunun (0,830) ve logitlerin ortalamasıyla yapılan geç füzyonun (0,833) test AUC değerlerini ve test hastalarından çekilen bootstrap örneklerinde erken füzyonun tek başına laboratuvardan daha iyi olduğu örneklerin oranını (0,041) bulacaksınız.

In [ ]:
#@title 1. durağın hazırlığı: Bu hücreyi bir kez çalıştırın (durağın verilerini yükler)
if "registry" not in globals():
    registry = followup.add_followup(cohort.make_cohort(3600))
    # 24 ay içinde bir olay; olay olmadan 24 aydan kısa izlenen hastalar dışarıda bırakılır
    registry = registry[(registry["event"] == 1) | (registry["followup_months"] >= 24)].reset_index(drop=True)
    registry["event24"] = ((registry["event"] == 1) & (registry["followup_months"] <= 24)).astype(int)
    number = registry["patient_id"].str[3:].astype(int)
    registry["set"] = np.select([number <= 2160, number <= 2880], ["training", "validation"], "test")
    registry["lbbb"] = (registry["ecg_conduction"] == "LBBB").astype(int)
    registry["af_ecg"] = (registry["ecg_rhythm"] == "AF").astype(int)
    registry["log_ntprobnp"] = np.log(registry["ntprobnp"])
    registry["log_troponin"] = np.log(registry["troponin"])
    HF_GROUPS = {
        "ECG": ["ecg_hr", "ecg_qrs", "ecg_qt", "lbbb", "af_ecg"],
        "eko": ["lvef", "lvedd_mm", "lavi", "e_e_prime", "tapse_mm", "lv_mass_index"],
        "laboratuvar": ["log_ntprobnp", "egfr", "log_troponin"],
    }
_SETS = {"training": "eğitim", "validation": "doğrulama", "test": "test"}


def _registry_table():
    """Kayıt sisteminin hastaları; kümeleri, üç modalitenin değerleri ve 24 ay içindeki olayla birlikte."""
    r = registry
    return pd.DataFrame({"hasta": r["patient_id"], "küme": r["set"].map(_SETS),
                         "kalp hızı": r["ecg_hr"], "QRS": r["ecg_qrs"], "QT": r["ecg_qt"],
                         "sol dal bloğu": r["lbbb"], "atriyal fibrilasyon": r["af_ecg"],
                         "ejeksiyon fraksiyonu": r["lvef"], "diyastol sonu çap": r["lvedd_mm"],
                         "sol atriyum hacim indeksi": r["lavi"], "E/e'": r["e_e_prime"], "TAPSE": r["tapse_mm"],
                         "sol ventrikül kütle indeksi": r["lv_mass_index"],
                         "NT-proBNP": r["ntprobnp"], "eGFR": r["egfr"], "troponin": r["troponin"], "olay": r["event24"]})


print(len(registry), "hasta,", int(registry["event24"].sum()), "hastada 24 ay içinde olay")
print("1. durak hazır.")

### Adım 1.1: Tek başına laboratuvar değerleri

**Amaç.** Bu adımda üç laboratuvar değeriyle kalp yetersizliği olayları için ilk model kurulur.

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter Notebook'uma yapıştırıp çalıştıracağım.  
> Bağlam: NT-proBNP, eGFR ve troponinin tek başına 24 ay içinde ölümü ya da kalp yetersizliği nedeniyle hastaneye yatışı ne kadar iyi öngördüğünü, EKG ve ekokardiyogramı ekleyecek modeller için bir başlangıç noktası olarak öğrenmek istiyorum.  
> Veri: Jupyter Notebook'ta "veri" adlı bir tablo vardır. Tabloda sanal kliniğin kayıt sisteminden, yani sentetik bir kohorttan 3.227 hasta bulunur: En az 24 ay izlenen ya da daha önce olay görülen hastalar. Her satır bir hastadır: "hasta" hastayı, "küme" kümesini verir ("eğitim", 1.936 hasta; "doğrulama", 640 hasta; "test", 651 hasta); "olay" ise hasta 24 ay içinde öldüyse ya da kalp yetersizliği nedeniyle hastaneye yattıysa 1, aksi hâlde 0 olur. EKG; "kalp hızı" (dakikada), "QRS" ve "QT" (ms), "sol dal bloğu" ve "atriyal fibrilasyon" (varsa 1) sütunlarını verir. Ekokardiyogram; "ejeksiyon fraksiyonu" (%), "diyastol sonu çap" (mm), "sol atriyum hacim indeksi" (mL/m²), "E/e'", "TAPSE" (mm) ve "sol ventrikül kütle indeksi" (g/m²) sütunlarını verir. Laboratuvar; "NT-proBNP" (pg/mL), "eGFR" (mL/dk/1,73 m²) ve "troponin" (yüksek duyarlıklı, ng/L) sütunlarını verir.  
> Görev: NT-proBNP ve troponin çok çarpık dağıldığı için ikisinin de logaritmasını kullan. Üç laboratuvar değerini öznitelik olarak kullan. Sütunları eğitim hastalarıyla standartlaştır ve eğitim hastalarında bir lojistik regresyon kur. Düzenlileştirmenin miktarını doğrulama hastalarındaki ROC eğrisi altındaki alana (AUC) göre seç; gücünün tersi için 0,01, 0,1, 1 ve 10 değerlerini dene ve seçilen modelin test hastalarındaki AUC değerini hesapla.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: Seçilen gücü, doğrulama ve test AUC değerlerini yazdır; test AUC değerini üç ondalık basamakla "cevap" adıyla sakla.  
> Test: AUC 0 ile 1 arasında olmalı ve hiçbir test hastası son adımdan önce kullanılmamalı.  

In [ ]:
#@title Adım 1.1 için veri: Bu hücreyi çalıştırın ("veri" tablosunu hazırlar ve gösterir)
veri = _registry_table()
globals().pop("cevap", None)
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

In [ ]:
# Adım 1.1 kontrolü
teach.check_step("Adım 1.1", "cevap", globals())

In [ ]:
#@title Adım 1.1 için hazır kod (yalnızca asistanın kodu kontrolden geçmezse çalıştırın)
veri = _registry_table()
d = veri.copy()
d["log NT-proBNP"] = np.log(d["NT-proBNP"])
d["log troponin"] = np.log(d["troponin"])
gruplar = {"ECG": ["kalp hızı", "QRS", "QT", "sol dal bloğu", "atriyal fibrilasyon"],
                  "eko": ["ejeksiyon fraksiyonu", "diyastol sonu çap", "sol atriyum hacim indeksi", "E/e'", "TAPSE", "sol ventrikül kütle indeksi"],
                  "laboratuvar": ["log NT-proBNP", "eGFR", "log troponin"]}
kumeler = d["küme"].to_numpy()
eg_, do_, te_ = kumeler == "eğitim", kumeler == "doğrulama", kumeler == "test"
y = d["olay"].to_numpy()


def kur(sutunlar):
    X = d[sutunlar].to_numpy(dtype=float)
    en_iyi = None
    for C in (0.01, 0.1, 1.0, 10.0):
        model = make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=5000)).fit(X[eg_], y[eg_])
        auc_dog = roc_auc_score(y[do_], model.predict_proba(X[do_])[:, 1])
        if en_iyi is None or auc_dog > en_iyi[0]:
            en_iyi = (auc_dog, C, model)
    return en_iyi[2].predict_proba(X)[:, 1]

p_lab = kur(gruplar["laboratuvar"])
print("test AUC", round(roc_auc_score(y[te_], p_lab[te_]), 3))
cevap = round(float(roc_auc_score(y[te_], p_lab[te_])), 3)
teach.check_step("Adım 1.1", "cevap", globals())

**Sonuç ne gösteriyor?** Üç laboratuvar değeri 0,849 test AUC değeri verir. Bunun çoğunu, gerilim altındaki yetersiz ventrikülün salgıladığı bir peptit olan NT-proBNP taşır.

### Adım 1.2: Üç modalitenin erken füzyonu

**Amaç.** Bu adımda üç modalitenin 14 değeri yan yana konur ve bunlar üzerinde tek bir model kurulur.

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter Notebook'uma yapıştırıp çalıştıracağım.  
> Bağlam: Bir hastanın EKG'sini, ekokardiyogramını ve laboratuvar değerlerini birlikte gören bir modelin kalp yetersizliği olaylarını tek başına laboratuvar değerlerinden daha iyi öngörüp öngörmediğini öğrenmek istiyorum.  
> Veri: Jupyter Notebook'ta "veri" adlı bir tablo vardır. Tabloda sanal kliniğin kayıt sisteminden, yani sentetik bir kohorttan 3.227 hasta bulunur: En az 24 ay izlenen ya da daha önce olay görülen hastalar. Her satır bir hastadır: "hasta" hastayı, "küme" kümesini verir ("eğitim", 1.936 hasta; "doğrulama", 640 hasta; "test", 651 hasta); "olay" ise hasta 24 ay içinde öldüyse ya da kalp yetersizliği nedeniyle hastaneye yattıysa 1, aksi hâlde 0 olur. EKG; "kalp hızı" (dakikada), "QRS" ve "QT" (ms), "sol dal bloğu" ve "atriyal fibrilasyon" (varsa 1) sütunlarını verir. Ekokardiyogram; "ejeksiyon fraksiyonu" (%), "diyastol sonu çap" (mm), "sol atriyum hacim indeksi" (mL/m²), "E/e'", "TAPSE" (mm) ve "sol ventrikül kütle indeksi" (g/m²) sütunlarını verir. Laboratuvar; "NT-proBNP" (pg/mL), "eGFR" (mL/dk/1,73 m²) ve "troponin" (yüksek duyarlıklı, ng/L) sütunlarını verir.  
> Görev: NT-proBNP ve troponin çok çarpık dağıldığı için ikisinin de logaritmasını kullan. Üç modalite EKG (kalp hızı, QRS, QT, sol dal bloğu, atriyal fibrilasyon), ekokardiyogram (ejeksiyon fraksiyonu, diyastol sonu çap, sol atriyum hacim indeksi, E/e', TAPSE, sol ventrikül kütle indeksi) ve laboratuvardır (NT-proBNP, eGFR, troponin). 14 değerin tamamını tek bir modelin öznitelikleri olarak birlikte kullan. Sütunları eğitim hastalarıyla standartlaştır ve eğitim hastalarında bir lojistik regresyon kur. Düzenlileştirmenin miktarını doğrulama hastalarındaki ROC eğrisi altındaki alana (AUC) göre seç; gücünün tersi için 0,01, 0,1, 1 ve 10 değerlerini dene ve seçilen modelin test hastalarındaki AUC değerini hesapla.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: Seçilen gücü, doğrulama ve test AUC değerlerini yazdır; test AUC değerini üç ondalık basamakla "cevap" adıyla sakla.  
> Test: Model 14 öznitelik kullanmalı ve AUC 0 ile 1 arasında olmalı.  

In [ ]:
#@title Adım 1.2 için veri: Bu hücreyi çalıştırın ("veri" tablosunu hazırlar ve gösterir)
veri = _registry_table()
globals().pop("cevap", None)
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

In [ ]:
# Adım 1.2 kontrolü
teach.check_step("Adım 1.2", "cevap", globals())

In [ ]:
#@title Adım 1.2 için hazır kod (yalnızca asistanın kodu kontrolden geçmezse çalıştırın)
veri = _registry_table()
d = veri.copy()
d["log NT-proBNP"] = np.log(d["NT-proBNP"])
d["log troponin"] = np.log(d["troponin"])
gruplar = {"ECG": ["kalp hızı", "QRS", "QT", "sol dal bloğu", "atriyal fibrilasyon"],
                  "eko": ["ejeksiyon fraksiyonu", "diyastol sonu çap", "sol atriyum hacim indeksi", "E/e'", "TAPSE", "sol ventrikül kütle indeksi"],
                  "laboratuvar": ["log NT-proBNP", "eGFR", "log troponin"]}
kumeler = d["küme"].to_numpy()
eg_, do_, te_ = kumeler == "eğitim", kumeler == "doğrulama", kumeler == "test"
y = d["olay"].to_numpy()


def kur(sutunlar):
    X = d[sutunlar].to_numpy(dtype=float)
    en_iyi = None
    for C in (0.01, 0.1, 1.0, 10.0):
        model = make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=5000)).fit(X[eg_], y[eg_])
        auc_dog = roc_auc_score(y[do_], model.predict_proba(X[do_])[:, 1])
        if en_iyi is None or auc_dog > en_iyi[0]:
            en_iyi = (auc_dog, C, model)
    return en_iyi[2].predict_proba(X)[:, 1]

p_erken = kur(sum(gruplar.values(), []))
print("öznitelik", len(sum(gruplar.values(), [])), "| test AUC", round(roc_auc_score(y[te_], p_erken[te_]), 3))
cevap = round(float(roc_auc_score(y[te_], p_erken[te_])), 3)
teach.check_step("Adım 1.2", "cevap", globals())

**Sonuç ne gösteriyor?** Erken füzyon 0,830 test AUC değeri verir; tek başına laboratuvarın değeri ise 0,849 düzeyindedir. On bir değer daha eklemek burada modeli iyileştirmemiştir.

### Adım 1.3: Logitlerin ortalamasıyla geç füzyon

**Amaç.** Bu adımda her modalite için bir model kurulur ve tahminleri logitlerinin ortalamasıyla birleştirilir.

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter Notebook'uma yapıştırıp çalıştıracağım.  
> Bağlam: Bütün değerler üzerinde kurulan tek bir modeli, her modalite için birer tane olmak üzere görüşlerinin ortalaması alınan üç ayrı modelle karşılaştırmak istiyorum.  
> Veri: Jupyter Notebook'ta "veri" adlı bir tablo vardır. Tabloda sanal kliniğin kayıt sisteminden, yani sentetik bir kohorttan 3.227 hasta bulunur: En az 24 ay izlenen ya da daha önce olay görülen hastalar. Her satır bir hastadır: "hasta" hastayı, "küme" kümesini verir ("eğitim", 1.936 hasta; "doğrulama", 640 hasta; "test", 651 hasta); "olay" ise hasta 24 ay içinde öldüyse ya da kalp yetersizliği nedeniyle hastaneye yattıysa 1, aksi hâlde 0 olur. EKG; "kalp hızı" (dakikada), "QRS" ve "QT" (ms), "sol dal bloğu" ve "atriyal fibrilasyon" (varsa 1) sütunlarını verir. Ekokardiyogram; "ejeksiyon fraksiyonu" (%), "diyastol sonu çap" (mm), "sol atriyum hacim indeksi" (mL/m²), "E/e'", "TAPSE" (mm) ve "sol ventrikül kütle indeksi" (g/m²) sütunlarını verir. Laboratuvar; "NT-proBNP" (pg/mL), "eGFR" (mL/dk/1,73 m²) ve "troponin" (yüksek duyarlıklı, ng/L) sütunlarını verir.  
> Görev: NT-proBNP ve troponin çok çarpık dağıldığı için ikisinin de logaritmasını kullan. Üç modalite EKG (kalp hızı, QRS, QT, sol dal bloğu, atriyal fibrilasyon), ekokardiyogram (ejeksiyon fraksiyonu, diyastol sonu çap, sol atriyum hacim indeksi, E/e', TAPSE, sol ventrikül kütle indeksi) ve laboratuvardır (NT-proBNP, eGFR, troponin). Her modalite için ayrı ayrı: sütunları eğitim hastalarıyla standartlaştır ve eğitim hastalarında bir lojistik regresyon kur. Düzenlileştirmenin miktarını doğrulama hastalarındaki ROC eğrisi altındaki alana (AUC) göre seç; gücünün tersi için 0,01, 0,1, 1 ve 10 değerlerini dene ve seçilen modelin test hastalarındaki AUC değerini hesapla. Ardından her hastanın üç olasılığını logite, log(p / (1 − p)), dönüştür, ortalamalarını al, ortalamayı yeniden olasılığa çevir ve test hastalarındaki AUC değerini hesapla.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: Her modalitenin ve geç füzyonlarının test AUC değerini yazdır; geç füzyonun AUC değerini üç ondalık basamakla "cevap" adıyla sakla.  
> Test: Her modalite için bir tane olmak üzere üç model olmalı ve AUC 0 ile 1 arasında olmalı.  

In [ ]:
#@title Adım 1.3 için veri: Bu hücreyi çalıştırın ("veri" tablosunu hazırlar ve gösterir)
veri = _registry_table()
globals().pop("cevap", None)
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

In [ ]:
# Adım 1.3 kontrolü
teach.check_step("Adım 1.3", "cevap", globals())

In [ ]:
#@title Adım 1.3 için hazır kod (yalnızca asistanın kodu kontrolden geçmezse çalıştırın)
veri = _registry_table()
d = veri.copy()
d["log NT-proBNP"] = np.log(d["NT-proBNP"])
d["log troponin"] = np.log(d["troponin"])
gruplar = {"ECG": ["kalp hızı", "QRS", "QT", "sol dal bloğu", "atriyal fibrilasyon"],
                  "eko": ["ejeksiyon fraksiyonu", "diyastol sonu çap", "sol atriyum hacim indeksi", "E/e'", "TAPSE", "sol ventrikül kütle indeksi"],
                  "laboratuvar": ["log NT-proBNP", "eGFR", "log troponin"]}
kumeler = d["küme"].to_numpy()
eg_, do_, te_ = kumeler == "eğitim", kumeler == "doğrulama", kumeler == "test"
y = d["olay"].to_numpy()


def kur(sutunlar):
    X = d[sutunlar].to_numpy(dtype=float)
    en_iyi = None
    for C in (0.01, 0.1, 1.0, 10.0):
        model = make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=5000)).fit(X[eg_], y[eg_])
        auc_dog = roc_auc_score(y[do_], model.predict_proba(X[do_])[:, 1])
        if en_iyi is None or auc_dog > en_iyi[0]:
            en_iyi = (auc_dog, C, model)
    return en_iyi[2].predict_proba(X)[:, 1]

logitler = []
for ad, sutunlar in gruplar.items():
    p_m = kur(sutunlar)
    print(ad, round(roc_auc_score(y[te_], p_m[te_]), 3))
    logitler.append(np.log(p_m / (1 - p_m)))
p_gec = 1 / (1 + np.exp(-np.mean(logitler, axis=0)))
print("geç füzyon", round(roc_auc_score(y[te_], p_gec[te_]), 3))
cevap = round(float(roc_auc_score(y[te_], p_gec[te_])), 3)
teach.check_step("Adım 1.3", "cevap", globals())

**Sonuç ne gösteriyor?** Üç logitin ortalaması 0,833 verir; bu değer erken füzyona (0,830) yakındır. Bilginin çoğunu tek bir modalite taşıdığında füzyonun yolu pek önemli olmaz.

### Adım 1.4: Farktan ne kadar emin olabiliriz?

**Amaç.** Bu adımda test hastaları yerine koyarak tekrar tekrar çekilir ve erken füzyonun tek başına laboratuvarı ne sıklıkla geçtiği görülür.

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter Notebook'uma yapıştırıp çalıştıracağım.  
> Bağlam: Az sayıda olayı olan birkaç yüz test hastasında ölçülen, üç modaliteli bir model ile tek başına laboratuvar değerleri arasındaki farkın başka hastalarda da geçerli olup olmayacağını öğrenmek istiyorum.  
> Veri: Jupyter Notebook'ta "veri" adlı bir tablo vardır. Tabloda sanal kliniğin kayıt sisteminden, yani sentetik bir kohorttan 3.227 hasta bulunur: En az 24 ay izlenen ya da daha önce olay görülen hastalar. Her satır bir hastadır: "hasta" hastayı, "küme" kümesini verir ("eğitim", 1.936 hasta; "doğrulama", 640 hasta; "test", 651 hasta); "olay" ise hasta 24 ay içinde öldüyse ya da kalp yetersizliği nedeniyle hastaneye yattıysa 1, aksi hâlde 0 olur. EKG; "kalp hızı" (dakikada), "QRS" ve "QT" (ms), "sol dal bloğu" ve "atriyal fibrilasyon" (varsa 1) sütunlarını verir. Ekokardiyogram; "ejeksiyon fraksiyonu" (%), "diyastol sonu çap" (mm), "sol atriyum hacim indeksi" (mL/m²), "E/e'", "TAPSE" (mm) ve "sol ventrikül kütle indeksi" (g/m²) sütunlarını verir. Laboratuvar; "NT-proBNP" (pg/mL), "eGFR" (mL/dk/1,73 m²) ve "troponin" (yüksek duyarlıklı, ng/L) sütunlarını verir.  
> Görev: NT-proBNP ve troponin çok çarpık dağıldığı için ikisinin de logaritmasını kullan. Üç modalite EKG (kalp hızı, QRS, QT, sol dal bloğu, atriyal fibrilasyon), ekokardiyogram (ejeksiyon fraksiyonu, diyastol sonu çap, sol atriyum hacim indeksi, E/e', TAPSE, sol ventrikül kütle indeksi) ve laboratuvardır (NT-proBNP, eGFR, troponin). Laboratuvar değerlerinin modelini ve 14 değerin tamamının modelini şöyle kur. Sütunları eğitim hastalarıyla standartlaştır ve eğitim hastalarında bir lojistik regresyon kur. Düzenlileştirmenin miktarını doğrulama hastalarındaki ROC eğrisi altındaki alana (AUC) göre seç; gücünün tersi için 0,01, 0,1, 1 ve 10 değerlerini dene ve seçilen modelin test hastalarındaki AUC değerini hesapla. Ardından 2.000 bootstrap örneği çek; her örnek test hastaları arasından yerine koyarak test hastası sayısı kadar hasta çeksin. Sonucun tekrarlanabilmesi için seed değerini 2026 olarak sabitle. Olay görülen ve görülmeyen hastaları içeren her örnekte iki modelin AUC değerini ve farkı (erken füzyon eksi laboratuvar) hesapla. Ortalama farkı, 2,5. ve 97,5. yüzdelikler arasındaki yüzde 95 aralığını ve farkın 0'ın üstünde olduğu örneklerin oranını bildir.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: Ortalama farkı ve aralığını yazdır; oranı üç ondalık basamakla "cevap" adıyla sakla.  
> Test: Oran 0 ile 1 arasında olmalı ve iki model her örnekte aynı hastalar üzerinde karşılaştırılmalı.  

In [ ]:
#@title Adım 1.4 için veri: Bu hücreyi çalıştırın ("veri" tablosunu hazırlar ve gösterir)
veri = _registry_table()
globals().pop("cevap", None)
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

In [ ]:
# Adım 1.4 kontrolü
teach.check_step("Adım 1.4", "cevap", globals())

In [ ]:
#@title Adım 1.4 için hazır kod (yalnızca asistanın kodu kontrolden geçmezse çalıştırın)
veri = _registry_table()
d = veri.copy()
d["log NT-proBNP"] = np.log(d["NT-proBNP"])
d["log troponin"] = np.log(d["troponin"])
gruplar = {"ECG": ["kalp hızı", "QRS", "QT", "sol dal bloğu", "atriyal fibrilasyon"],
                  "eko": ["ejeksiyon fraksiyonu", "diyastol sonu çap", "sol atriyum hacim indeksi", "E/e'", "TAPSE", "sol ventrikül kütle indeksi"],
                  "laboratuvar": ["log NT-proBNP", "eGFR", "log troponin"]}
kumeler = d["küme"].to_numpy()
eg_, do_, te_ = kumeler == "eğitim", kumeler == "doğrulama", kumeler == "test"
y = d["olay"].to_numpy()


def kur(sutunlar):
    X = d[sutunlar].to_numpy(dtype=float)
    en_iyi = None
    for C in (0.01, 0.1, 1.0, 10.0):
        model = make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=5000)).fit(X[eg_], y[eg_])
        auc_dog = roc_auc_score(y[do_], model.predict_proba(X[do_])[:, 1])
        if en_iyi is None or auc_dog > en_iyi[0]:
            en_iyi = (auc_dog, C, model)
    return en_iyi[2].predict_proba(X)[:, 1]

p_lab = kur(gruplar["laboratuvar"])
p_erken = kur(sum(gruplar.values(), []))
yt, a_, b_ = y[te_], p_erken[te_], p_lab[te_]
rng = np.random.default_rng(2026)
fark = []
while len(fark) < 2000:
    i = rng.integers(0, len(yt), len(yt))
    if 0 < yt[i].sum() < len(i):
        fark.append(roc_auc_score(yt[i], a_[i]) - roc_auc_score(yt[i], b_[i]))
fark = np.array(fark)
print("ortalama fark", round(fark.mean(), 3), "| yüzde 95 aralığı", np.round(np.percentile(fark, [2.5, 97.5]), 3))
cevap = round(float((fark > 0).mean()), 3)
teach.check_step("Adım 1.4", "cevap", globals())

**Sonuç ne gösteriyor?** Erken füzyon örneklerin 0,041 kadarında tek başına laboratuvardan daha iyidir. Birkaç düzine test olayıyla bu büyüklükteki bir fark her iki yöne de gidebilir; NT-proBNP öteki iki modalitenin gösterdiklerinin önemli bir bölümünü zaten özetlemektedir.

**1. durak tamamlandı.** [Ders sayfasına dönün](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-08/ders.html#last-two-hours-cardiovascular-risk-scoring-system) ve okumaya «Son 2 saat: Kardiyovasküler risk skorlama sistemi» bölümünden devam edin. Bir sonraki uygulama durağında ders sayfası sizi yine bu Jupyter Notebook'a getirir.

## Durak 2: Göğüs ağrısı ünitesinde majör olumsuz kardiyak olaylar: HEART, tek modaliteler ve füzyon

*Ders: [2. uygulama durağı](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-08/ders.html#stop-2)* · *Python yolunda aynı sonuç: Kısım 2, 4-8. bölümler*

Sanal kliniğin göğüs ağrısı ünitesi 30 gün içindeki MACE'i öngörür. Çıtayı HEART skoru belirler. Bir model klinik tabloyu, troponinleri ve ekokardiyogramı birleştirir; ikinci bir model beş tek modaliteli modeli istifler; son adım ise iki tür geç füzyonun olasılıklarına güvenilip güvenilemeyeceğini sorar.

**Durağın sonunda:** HEART skorunun (0,903), klinik tablonun, troponinlerin ve ekokardiyogramın erken füzyonunun (0,900) ve beş tek modaliteli modelin istiflenmesinin (0,922) test AUC değerlerini ve logitlerin ortalamasıyla yapılan geç füzyonun kalibrasyon eğimini (2,78) bulacaksınız.

In [ ]:
#@title 2. durağın hazırlığı: Bu hücreyi bir kez çalıştırın (durağın verilerini yükler)
if "chest" not in globals():
    chest = pd.concat([triage.presentations("A"), triage.presentations("B")], ignore_index=True)
    A = chest[chest["hospital"] == "A"].reset_index(drop=True)
    B = chest[chest["hospital"] == "B"].reset_index(drop=True)
    tr, va, te = [(A["set"] == s).to_numpy() for s in ("training", "validation", "test")]
    y, yB = A["mace30"].to_numpy(), B["mace30"].to_numpy()
if "fit_logistic" not in globals():
    def fit_logistic(X, y, Cs=(0.01, 0.1, 1.0, 10.0), Xv=None, yv=None):
        """Sütunları standartlaştırır ve bir lojistik regresyon kurar; doğrulama kümesi verilirse doğrulama AUC değeri en büyük olan ters düzenlileştirme gücünü tutar."""
        best = None
        for C in (Cs if Xv is not None else (1.0,)):
            model = make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=5000)).fit(X, y)
            auc = roc_auc_score(yv, model.predict_proba(Xv)[:, 1]) if Xv is not None else 0.0
            if best is None or auc > best[1]:
                best = (model, auc)
        return best[0]


    def logit(p):
        """Bir olasılığın log-odds değerini (logit) verir."""
        p = np.clip(p, 1e-6, 1 - 1e-6)
        return np.log(p / (1 - p))


    def bootstrap_auc(y, p, n=2000, seed=2026):
        """Hastalardan çekilen n bootstrap örneğinde AUC: n değerden oluşan bir dizi."""
        rng = np.random.default_rng(seed)
        out = []
        while len(out) < n:
            i = rng.integers(0, len(y), len(y))
            if 0 < y[i].sum() < len(i):
                out.append(roc_auc_score(y[i], p[i]))
        return np.array(out)


    def paired_bootstrap(y, p1, p2, n=2000, seed=2026):
        """Aynı hastalardan çekilen n bootstrap örneğinde p1'in AUC değeri eksi p2'nin AUC değeri: n farktan oluşan bir dizi."""
        rng = np.random.default_rng(seed)
        out = []
        while len(out) < n:
            i = rng.integers(0, len(y), len(y))
            if 0 < y[i].sum() < len(i):
                out.append(roc_auc_score(y[i], p1[i]) - roc_auc_score(y[i], p2[i]))
        return np.array(out)


    def calibration(y, p):
        """Kalibrasyon sabiti (eğim 1'e sabitlenerek bulunur) ve kalibrasyon eğimi."""
        z = logit(p)
        a = 0.0
        for _ in range(50):
            q = 1 / (1 + np.exp(-(a + z)))
            a += (y - q).sum() / (q * (1 - q)).sum()
        slope = LogisticRegression(C=1e6, max_iter=5000).fit(z[:, None], y).coef_[0, 0]
        return a, slope


    def reliability(y, p, bins=10):
        """Eşit büyüklükte gruplarda ortalama tahmin edilen risk ve gözlenen olay oranı: İki dizi."""
        order = np.argsort(p)
        groups = np.array_split(order, bins)
        return np.array([p[g].mean() for g in groups]), np.array([y[g].mean() for g in groups])
_SETS = {"training": "eğitim", "validation": "doğrulama", "test": "test", "external": "B hastanesi"}
# kümeler ve sonuçlar, bir asistanın kodunun üzerine yazmadığı özel adlarla
_tr, _va, _te = [(A["set"] == _s).to_numpy() for _s in ("training", "validation", "test")]
_y, _yB = A["mace30"].to_numpy(), B["mace30"].to_numpy()


def _unit_table(d):
    """Bir hastanenin başvuruları; ekibin geliş anında bildikleri ve 30 gün içindeki MACE ile birlikte."""
    url = d["troponin_url"]
    return pd.DataFrame({"başvuru": d["patient_id"], "küme": d["set"].map(_SETS), "yaş": d["age"],
                         "cinsiyet": d["sex"].map({"M": "erkek", "F": "kadın"}),
                         "sigara": d["smoker"], "diyabet": d["diabetes"], "hipertansiyon": d["hypertension"],
                         "hiperlipidemi": d["hyperlipidaemia"], "obezite": d["obesity"],
                         "aile öyküsü": d["family_history"], "bilinen koroner arter hastalığı": d["known_cad"],
                         "kalp hızı": d["heart_rate"], "sistolik basınç": d["sbp"], "eGFR": d["egfr"],
                         "gelişte troponin": (d["troponin_0h"] / url).round(2),
                         "bir saat sonra troponin": (d["troponin_1h"] / url).round(2),
                         "ekokardiyogram yapıldı": d["echo_done"], "ejeksiyon fraksiyonu": d["lvef"],
                         "duvar hareket bozukluğu": d["rwma"], "HEART": d["heart"], "MACE": d["mace30"]})


def _course_probabilities():
    """Dersin modellerinin A ve B hastanelerinde verdiği MACE olasılıkları: Beş tek modalite, erken füzyon ve logitlerin ortalaması (bir kez hesaplanır, bir ile üç dakika sürer; sürenin çoğu EKG ağı içindir)."""
    if "_prob_cache" in globals():
        return _prob_cache
    tr, va, y = _tr, _va, _y
    CLINICAL = ["age", "male", "smoker", "diabetes", "hypertension", "hyperlipidaemia", "obesity", "family_history", "known_cad",
                "heart_rate", "sbp", "egfr"]


    def clinical(d):
        """Klinik tablo: Yaş, cinsiyet, risk faktörleri, bilinen koroner arter hastalığı, kalp hızı, sistolik basınç ve eGFR."""
        return d.assign(male=(d["sex"] == "M").astype(int))[CLINICAL].astype(float)


    def laboratory(d):
        """Kendi testinin üst referans sınırının katları olarak troponin: Gelişteki ve bir saat sonraki değerin logaritması, bir saatteki değişim ve ikinci değerin olup olmadığı."""
        url = d["troponin_url"]
        t0, t1 = d["troponin_0h"] / url, d["troponin_1h"] / url
        return pd.DataFrame({"log_trop_0h": np.log(t0), "log_trop_1h": np.log(t1.fillna(t0)),
                             "delta_1h": (t1 - t0).fillna(0.0), "has_1h": t1.notna().astype(float)})


    def echo(d):
        """Yatak başı ekokardiyogram: Yapılıp yapılmadığı, ejeksiyon fraksiyonu (yapılmadıysa 60) ve bölgesel duvar hareket bozukluğu (yapılmadıysa 0)."""
        return pd.DataFrame({"echo_done": d["echo_done"].astype(float), "lvef": d["lvef"].fillna(60.0), "rwma": d["rwma"].fillna(0.0)})

    tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=3, sublinear_tf=True).fit(A.loc[tr, "note_tr"])
    svd = TruncatedSVD(n_components=20, random_state=RANDOM_STATE).fit(tfidf.transform(A.loc[tr, "note_tr"]))


    def note(d):
        """Triyaj notu 20 sayı olarak: Kelimelerin ve kelime çiftlerinin TF-IDF değerleri, eğitim notlarında öğrenilen bir tekil değer ayrışımıyla küçültülür."""
        return pd.DataFrame(svd.transform(tfidf.transform(d["note_tr"])), columns=[f"note_{i + 1}" for i in range(20)])

    ecg_A = triage.ecg_array(A)          # 12 derivasyonlu 4.000 EKG, 100 Hz'de 10 saniye: 4000 x 12 x 1000
    ecg_B = triage.ecg_array(B)


    class ECGNet(nn.Module):
        """12 derivasyonlu bir EKG için tek boyutlu evrişimli ağ: Dört blok, her kanalın zaman boyunca ortalaması ve tek bir çıktı."""
        def __init__(self):
            super().__init__()
            block = lambda a, b: nn.Sequential(nn.Conv1d(a, b, 7, padding=3), nn.BatchNorm1d(b), nn.ReLU(), nn.MaxPool1d(2))
            self.features = nn.Sequential(block(12, 32), block(32, 64), block(64, 64), block(64, 64))
            self.head = nn.Linear(64, 1)

        def embed(self, x):
            return self.features(x).mean(dim=2)

        def forward(self, x):
            return self.head(self.embed(x)).squeeze(1)


    def train_ecg(model, X, y, Xv, yv, epochs=12, seed=2026, verbose=True):
        """64 EKG'lik gruplarla, öğrenme hızı 0,001 olan Adam; her EKG zamanda en fazla yarım saniye kaydırılır. Doğrulama AUC değeri en iyi olan epoch'un ağırlıkları saklanır."""
        torch.manual_seed(seed)
        X, y = torch.as_tensor(X), torch.as_tensor(y, dtype=torch.float32)
        optimiser = torch.optim.Adam(model.parameters(), lr=0.001)
        loss_fn = nn.BCEWithLogitsLoss()
        best, state = -1.0, None
        for epoch in range(epochs):
            model.train()
            order = torch.randperm(len(X))
            for i in range(0, len(X), 64):
                b = order[i:i + 64]
                xb = torch.roll(X[b], int(torch.randint(-50, 51, (1,))), dims=2)
                loss = loss_fn(model(xb), y[b])
                optimiser.zero_grad()
                loss.backward()
                optimiser.step()
            a = roc_auc_score(yv, ecg_predict(model, Xv))
            if a > best:
                best, state = a, copy.deepcopy(model.state_dict())
            if verbose:
                print(f"epoch {epoch + 1}: doğrulama AUC {a:.3f}")
        model.load_state_dict(state)
        return best


    @torch.no_grad()
    def ecg_predict(model, X, batch=256):
        model.eval()
        return torch.cat([torch.sigmoid(model(torch.as_tensor(X[i:i + batch]))) for i in range(0, len(X), batch)]).numpy()


    @torch.no_grad()
    def ecg_embed(model, X, batch=256):
        model.eval()
        return torch.cat([model.embed(torch.as_tensor(X[i:i + batch])) for i in range(0, len(X), batch)]).numpy()
    torch.manual_seed(RANDOM_STATE)
    net = ECGNet()
    train_ecg(net, ecg_A[tr], y[tr], ecg_A[va], y[va], verbose=False)
    emb = lambda X: pd.DataFrame(ecg_embed(net, X), columns=[f"ecg_{i + 1}" for i in range(64)])
    out = {}
    for name, d, X in (("A", A, ecg_A), ("B", B, ecg_B)):
        parts = {"klinik": clinical(d), "troponin": laboratory(d), "ekokardiyogram": echo(d), "not": note(d)}
        out[name] = dict(parts=parts, ecg=ecg_predict(net, X), embedding=emb(X))
    p = {"A": {}, "B": {}}
    for k in out["A"]["parts"]:
        m = fit_logistic(out["A"]["parts"][k][tr], y[tr], Xv=out["A"]["parts"][k][va], yv=y[va])
        for h in ("A", "B"):
            p[h][k] = m.predict_proba(out[h]["parts"][k])[:, 1]
    for h in ("A", "B"):
        p[h]["EKG ağı"] = out[h]["ecg"]
    early = lambda h: pd.concat([out[h]["parts"]["klinik"], out[h]["parts"]["troponin"],
                                 out[h]["parts"]["ekokardiyogram"], out[h]["embedding"], out[h]["parts"]["not"]], axis=1)
    m = fit_logistic(early("A")[tr], y[tr], Xv=early("A")[va], yv=y[va])
    for h in ("A", "B"):
        p[h]["early"] = m.predict_proba(early(h))[:, 1]
        z = np.column_stack([logit(p[h][k]) for k in ["klinik", "troponin", "ekokardiyogram", "EKG ağı", "not"]])
        p[h]["mean"] = 1 / (1 + np.exp(-z.mean(axis=1)))
    globals()["_prob_cache"] = p
    return p


print(len(A), "başvuru A hastanesinde,", len(B), "başvuru B hastanesinde")
print("2. durak hazır.")

### Adım 2.1: HEART skoru

**Amaç.** Bu adımda HEART skorunun beş öğesi toplanır ve toplamın MACE görülen ve görülmeyen başvuruları ne kadar iyi ayırdığı görülür.

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter Notebook'uma yapıştırıp çalıştıracağım.  
> Bağlam: Ünitenin her modelinin aşması gereken çıta olarak HEART skorunun göğüs ağrısı ünitesinin test başvurularındaki sonucunu öğrenmek istiyorum.  
> Veri: Jupyter Notebook'ta "veri" adlı bir tablo vardır. Tabloda sanal kliniğin bir göğüs ağrısı ünitesinden, yani sentetik bir veri setinden 800 test başvurusu bulunur. Her satır bir başvurudur: "başvuru" numarasını verir; "öykü", "EKG", "yaş", "risk faktörleri" ve "troponin" sütunları HEART skorunun beş öğesinin puanlarını (her biri 0, 1 ya da 2) verir; "MACE" ise 30 gün içinde majör olumsuz kardiyak olay (miyokard enfarktüsü, acil revaskülarizasyon ya da ölüm) görüldüyse 1, aksi hâlde 0 olur.  
> Görev: Beş öğeyi HEART skorunda (0-10) topla. Skorun "MACE" sütununa göre AUC değerini hesapla. Ardından 0-3, 4-6 ve 7-10 gruplarını oluştur ve her biri için başvuru sayısını, oranını ve gruptaki MACE oranını ver.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: Üç grubun tablosunu yazdır; AUC değerini üç ondalık basamakla "cevap" adıyla sakla.  
> Test: Her skor 0 ile 10 arasında olmalı ve üç grup 800 başvurunun tamamını içermeli.  

In [ ]:
#@title Adım 2.1 için veri: Bu hücreyi çalıştırın ("veri" tablosunu hazırlar ve gösterir)
_t = A[_te].reset_index(drop=True)
veri = pd.DataFrame({"başvuru": _t["patient_id"], "öykü": _t["heart_h"], "EKG": _t["heart_e"], "yaş": _t["heart_a"],
                     "risk faktörleri": _t["heart_r"], "troponin": _t["heart_t"], "MACE": _t["mace30"]})
globals().pop("cevap", None)
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

In [ ]:
# Adım 2.1 kontrolü
teach.check_step("Adım 2.1", "cevap", globals())

In [ ]:
#@title Adım 2.1 için hazır kod (yalnızca asistanın kodu kontrolden geçmezse çalıştırın)
_t = A[_te].reset_index(drop=True)
veri = pd.DataFrame({"başvuru": _t["patient_id"], "öykü": _t["heart_h"], "EKG": _t["heart_e"], "yaş": _t["heart_a"],
                     "risk faktörleri": _t["heart_r"], "troponin": _t["heart_t"], "MACE": _t["mace30"]})
heart = veri[["öykü", "EKG", "yaş", "risk faktörleri", "troponin"]].sum(axis=1)
grup = pd.cut(heart, [-1, 3, 6, 10], labels=["0-3", "4-6", "7-10"])
print(veri.assign(grup=grup).groupby("grup", observed=True)["MACE"].agg(["size", "mean"]).round(3))
cevap = round(float(roc_auc_score(veri["MACE"], heart)), 3)
teach.check_step("Adım 2.1", "cevap", globals())

**Sonuç ne gösteriyor?** HEART, her biri 0-2 puanlık beş öğeyle 0,903 test AUC değerine ulaşır. Grupları riski iyi ayırır: 0-3 puanda az, 7-10 puanda çok olay görülür.

### Adım 2.2: Klinik tablonun, troponinlerin ve ekokardiyogramın erken füzyonu

**Amaç.** Bu adımda her başvurunun klinik değerleri, troponinleri ve ekokardiyogramı tek bir satıra konur, boşluklar bir işaretle doldurulur ve tek bir model kurulur.

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter Notebook'uma yapıştırıp çalıştıracağım.  
> Bağlam: Ekibin gelişten bir saat sonra bildikleriyle, EKG ve not olmadan, ikinci troponini ya da ekokardiyogramı eksik olan hastaları da tutarak 30 gün içindeki MACE için bir risk modeli kurmak istiyorum.  
> Veri: Jupyter Notebook'ta "veri" adlı bir tablo vardır. Tabloda sanal kliniğin bir göğüs ağrısı ünitesinden, yani sentetik bir veri setinden göğüs ağrısıyla yapılmış 4.000 başvuru bulunur. Her satır bir başvurudur: "başvuru" numarasını, "küme" kümesini verir ("eğitim", 2.400; "doğrulama", 800; "test", 800). Klinik tablo; "yaş" (yıl), "cinsiyet" ("erkek" ya da "kadın"), varsa 1 olan yedi risk faktörü ("sigara", "diyabet", "hipertansiyon", "hiperlipidemi", "obezite", "aile öyküsü" ve "bilinen koroner arter hastalığı"), "kalp hızı" (dakikada), "sistolik basınç" (mmHg) ve "eGFR" sütunlarını verir. Yüksek duyarlıklı troponin T, testin üst referans sınırının katları olarak verilir: "gelişte troponin" ve ölçülmediğinde boş kalan "bir saat sonra troponin". Yatak başı ekokardiyogram "ekokardiyogram yapıldı" (1 ya da 0) sütununu ve yapıldıysa "ejeksiyon fraksiyonu" (%) ile "duvar hareket bozukluğu" (bölgesel bir bozukluk görüldüyse 1) sütunlarını verir; yapılmadıysa bu iki sütun boştur. "HEART" HEART skorudur; "MACE" ise 30 gün içinde majör olumsuz kardiyak olay görüldüyse 1, aksi hâlde 0 olur.  
> Görev: Her başvurunun özniteliklerini oluştur: Yaş, erkek için 1 olan cinsiyet, yedi risk faktörü, kalp hızı, sistolik basınç ve eGFR; gelişteki troponinin logaritması, bir saat sonraki troponinin logaritması (ölçülmediyse gelişteki değer), bir saatteki değişim (ölçülmediyse 0) ve ikinci troponin ölçüldüyse 1 olan bir sütun; ekokardiyogramın yapılıp yapılmadığı, ejeksiyon fraksiyonu (yapılmadıysa 60) ve duvar hareket bozukluğu (yapılmadıysa 0). HEART skorunu kullanma. Sütunları eğitim başvurularıyla standartlaştır ve eğitim başvurularında bir lojistik regresyon kur. Düzenlileştirmenin miktarını doğrulama başvurularındaki ROC eğrisi altındaki alana (AUC) göre seç; gücünün tersi için 0,01, 0,1, 1 ve 10 değerlerini dene ve seçilen modelin test başvurularındaki AUC değerini hesapla.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: Seçilen gücü, doğrulama ve test AUC değerlerini yazdır; test AUC değerini üç ondalık basamakla "cevap" adıyla sakla.  
> Test: 4.000 başvurunun tamamı tutulmalı, model 19 öznitelik kullanmalı ve HEART skoru bunların arasında olmamalı.  

In [ ]:
#@title Adım 2.2 için veri: Bu hücreyi çalıştırın ("veri" tablosunu hazırlar ve gösterir)
veri = _unit_table(A)
globals().pop("cevap", None)
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

In [ ]:
# Adım 2.2 kontrolü
teach.check_step("Adım 2.2", "cevap", globals())

In [ ]:
#@title Adım 2.2 için hazır kod (yalnızca asistanın kodu kontrolden geçmezse çalıştırın)
veri = _unit_table(A)
d = veri
kumeler = d["küme"].to_numpy()
eg_, do_, te_ = kumeler == "eğitim", kumeler == "doğrulama", kumeler == "test"
y = d["MACE"].to_numpy()
t0, t1 = d["gelişte troponin"], d["bir saat sonra troponin"]
X = pd.DataFrame({
    "yaş": d["yaş"], "erkek": (d["cinsiyet"] == "erkek").astype(int),
    **{s: d[s] for s in ["sigara", "diyabet", "hipertansiyon", "hiperlipidemi", "obezite",
                                   "aile öyküsü", "bilinen koroner arter hastalığı", "kalp hızı", "sistolik basınç", "eGFR"]},
    "log gelişte troponin": np.log(t0), "log bir saat sonra troponin": np.log(t1.fillna(t0)),
    "bir saatteki değişim": (t1 - t0).fillna(0.0), "ikinci troponin ölçüldü": t1.notna().astype(int),
    "ekokardiyogram yapıldı": d["ekokardiyogram yapıldı"],
    "ejeksiyon fraksiyonu": d["ejeksiyon fraksiyonu"].fillna(60.0),
    "duvar hareket bozukluğu": d["duvar hareket bozukluğu"].fillna(0.0)}).to_numpy(dtype=float)
en_iyi = None
for C in (0.01, 0.1, 1.0, 10.0):
    model = make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=5000)).fit(X[eg_], y[eg_])
    auc_dog = roc_auc_score(y[do_], model.predict_proba(X[do_])[:, 1])
    if en_iyi is None or auc_dog > en_iyi[0]:
        en_iyi = (auc_dog, C, model)
p = en_iyi[2].predict_proba(X)[:, 1]
print("ters düzenlileştirme gücü", en_iyi[1], "| doğrulama AUC", round(en_iyi[0], 3),
      "| test AUC", round(roc_auc_score(y[te_], p[te_]), 3))
cevap = round(float(roc_auc_score(y[te_], p[te_])), 3)
teach.check_step("Adım 2.2", "cevap", globals())

**Sonuç ne gösteriyor?** Tek bir satırda birleştirilen üç modalite 0,900 test AUC değeri verir; HEART skorunun değeri ise 0,903 düzeyindedir. Eksik değeri işaretleyen sütunlar modelin her hastayı tutmasını sağlar.

### Adım 2.3: Beş tek modaliteli modelin istiflenmesi

**Amaç.** Bu adımda beş tek modaliteli modelin her birinin ne kadar sayılması gerektiği doğrulama başvurularında öğrenilir ve birleşim sınanır.

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter Notebook'uma yapıştırıp çalıştıracağım.  
> Bağlam: Her biri göğüs ağrılı bir başvurunun bir modalitesini okuyan beş modelim var; bunların görüşlerini eşit bir oy yerine verilerden öğrenilen ağırlıklarla tek bir riskte birleştirmek istiyorum.  
> Veri: Jupyter Notebook'ta "veri" adlı bir tablo vardır. Tabloda sanal kliniğin bir göğüs ağrısı ünitesinden, yani sentetik bir veri setinden 800 doğrulama ve 800 test başvurusu bulunur. Her satır bir başvurudur: "küme" kümesini verir ("doğrulama" ya da "test"); beş sütun ise dersin, her biri tek bir modaliteden öğrenen ve hepsi eğitim başvurularıyla eğitilmiş beş modelinin 30 gün içinde majör olumsuz kardiyak olay için verdiği olasılıkları içerir: "olasılık, klinik", "olasılık, troponin", "olasılık, ekokardiyogram", "olasılık, EKG ağı" ve "olasılık, not". "MACE" böyle bir olay görüldüyse 1, aksi hâlde 0 olur.  
> Görev: Her başvurunun beş olasılığını logite, log(p / (1 − p)), dönüştür. Yalnızca doğrulama başvurularında, varsayılan düzenlileştirmeyle, MACE'in beş logit üzerine bir lojistik regresyonunu kur. Beş ağırlığını ve sabit terimini bildir ve olasılıklarının test başvurularındaki AUC değerini hesapla. Sonucu beş logitin ortalamasıyla karşılaştır.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: İstiflemenin ağırlıklarını, sabit terimini ve test AUC değerini ve ortalamanın test AUC değerini yazdır; istiflemenin test AUC değerini üç ondalık basamakla "cevap" adıyla sakla.  
> Test: İstifleme yalnızca doğrulama başvurularında öğrenmeli ve her ağırlık yazdırılmalı.  

In [ ]:
#@title Adım 2.3 için veri: Bu hücreyi çalıştırın ("veri" tablosunu hazırlar ve gösterir)
_p = _course_probabilities()["A"]   # ilk seferde bir ile üç dakika sürer: Dersin modelleri bir kez eğitilir
_k = _va | _te
veri = pd.DataFrame({"küme": np.where(_va, "doğrulama", "test")[_k],
                     **{"olasılık, " + _n: _p[_n][_k] for _n in ["klinik", "troponin", "ekokardiyogram", "EKG ağı", "not"]},
                     "MACE": _y[_k]})
globals().pop("cevap", None)
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

In [ ]:
# Adım 2.3 kontrolü
teach.check_step("Adım 2.3", "cevap", globals())

In [ ]:
#@title Adım 2.3 için hazır kod (yalnızca asistanın kodu kontrolden geçmezse çalıştırın)
_p = _course_probabilities()["A"]   # ilk seferde bir ile üç dakika sürer: Dersin modelleri bir kez eğitilir
_k = _va | _te
veri = pd.DataFrame({"küme": np.where(_va, "doğrulama", "test")[_k],
                     **{"olasılık, " + _n: _p[_n][_k] for _n in ["klinik", "troponin", "ekokardiyogram", "EKG ağı", "not"]},
                     "MACE": _y[_k]})
sutunlar = [s for s in veri.columns if s.startswith("olasılık")]
p = np.clip(veri[sutunlar].to_numpy(), 1e-6, 1 - 1e-6)
z = np.log(p / (1 - p))
y = veri["MACE"].to_numpy()
do_, te_ = veri["küme"].to_numpy() == "doğrulama", veri["küme"].to_numpy() == "test"
istif = LogisticRegression(max_iter=5000).fit(z[do_], y[do_])
print(dict(zip(sutunlar, np.round(istif.coef_[0], 3))), "| sabit terim", round(istif.intercept_[0], 3))
auc_ort = roc_auc_score(y[te_], z[te_].mean(axis=1))
auc_istif = roc_auc_score(y[te_], istif.predict_proba(z[te_])[:, 1])
print("logitlerin ortalaması", round(auc_ort, 3), "| istifleme", round(auc_istif, 3))
cevap = round(float(auc_istif), 3)
teach.check_step("Adım 2.3", "cevap", globals())

**Sonuç ne gösteriyor?** İstifleme 0,922 test AUC değerine ulaşır. En büyük ağırlığı troponin alır ve ağırlıkların toplamı 1'den büyüktür: İstifleme modalitelerin kanıtlarının ortalamasını almak yerine onları toplar.

### Adım 2.4: Olasılıklara güvenilebilir mi?

**Amaç.** Bu adımda AUC değerleri neredeyse aynı olan iki modelin, erken füzyonun ve logitlerin ortalamasının, kalibrasyonu değerlendirilir.

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter Notebook'uma yapıştırıp çalıştıracağım.  
> Bağlam: Hastalara MACE risklerini söylemek ve bir eşiğin altındakileri eve göndermek istiyorum; hastaları eşit derecede iyi sıralayan iki modelin olasılıklarının söylediklerini ifade edip etmediğini öğrenmek istiyorum.  
> Veri: Jupyter Notebook'ta "veri" adlı bir tablo vardır. Tabloda sanal kliniğin bir göğüs ağrısı ünitesinden, yani sentetik bir veri setinden 800 test başvurusu bulunur. Her satır bir başvurudur: "olasılık, erken füzyon", dersin beş modaliteyi birlikte kullanan bir modelinin 30 gün içinde majör olumsuz kardiyak olay için verdiği olasılık; "olasılık, logitlerin ortalaması", beş tek modaliteli modelin logitlerinin ortalamasından gelen olasılıktır. "MACE" böyle bir olay görüldüyse 1, aksi hâlde 0 olur.  
> Görev: İki olasılık sütununun her biri için AUC değerini ve ortalama tahmin edilen riski MACE oranının yanında hesapla. Kalibrasyon sabitini (eğim 1'e sabitlenmişken ortalama tahmin edilen riski gözlenen orana eşitleyen logit kayması), kalibrasyon eğimini (MACE'in olasılığın logiti üzerine düzenlileştirmesiz bir lojistik regresyonu) ve Brier skorunu hesapla. Ardından her biri için bir güvenilirlik eğrisi çiz: Başvuruları olasılığa göre sırala, eşit büyüklükte on gruba böl ve her grubun ortalama olasılığını MACE oranına karşı, köşegenle birlikte çiz.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: İki modelin AUC değerini, ortalama tahmin edilen riskini, kalibrasyon sabitini, eğimini ve Brier skorunu yazdır, iki eğriyi göster; logitlerin ortalamasının kalibrasyon eğimini iki ondalık basamakla "cevap" adıyla sakla.  
> Test: İki eğrinin her biri on noktadan oluşmalı ve Brier skorları 0 ile 1 arasında olmalı.  

In [ ]:
#@title Adım 2.4 için veri: Bu hücreyi çalıştırın ("veri" tablosunu hazırlar ve gösterir)
_p = _course_probabilities()["A"]   # ilk seferde bir ile üç dakika sürer
veri = pd.DataFrame({"olasılık, erken füzyon": _p["early"][_te], "olasılık, logitlerin ortalaması": _p["mean"][_te],
                     "MACE": _y[_te]})
globals().pop("cevap", None)
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

In [ ]:
# Adım 2.4 kontrolü
teach.check_step("Adım 2.4", "cevap", globals())

In [ ]:
#@title Adım 2.4 için hazır kod (yalnızca asistanın kodu kontrolden geçmezse çalıştırın)
_p = _course_probabilities()["A"]   # ilk seferde bir ile üç dakika sürer
veri = pd.DataFrame({"olasılık, erken füzyon": _p["early"][_te], "olasılık, logitlerin ortalaması": _p["mean"][_te],
                     "MACE": _y[_te]})


def kalibrasyon(yy, p):
    z = np.log(np.clip(p, 1e-6, 1 - 1e-6) / (1 - np.clip(p, 1e-6, 1 - 1e-6)))
    a = 0.0
    for _ in range(50):
        q = 1 / (1 + np.exp(-(a + z)))
        a += (yy - q).sum() / (q * (1 - q)).sum()
    egim = LogisticRegression(C=1e6, max_iter=5000).fit(z[:, None], yy).coef_[0, 0]
    return a, egim, float(np.mean((p - yy) ** 2))

y = veri["MACE"].to_numpy()
fig, ax = plt.subplots(figsize=(4.6, 4.2))
for sutun in ["olasılık, erken füzyon", "olasılık, logitlerin ortalaması"]:
    p = veri[sutun].to_numpy()
    a, e, b = kalibrasyon(y, p)
    print(sutun, "| AUC", round(roc_auc_score(y, p), 3), "| ortalama", round(p.mean(), 3), "| gözlenen", round(y.mean(), 3),
          "| kalibrasyon sabiti", round(a, 2), "| eğim", round(e, 2), "| Brier", round(b, 3))
    gruplar = np.array_split(np.argsort(p), 10)
    ax.plot([p[g].mean() for g in gruplar], [y[g].mean() for g in gruplar], "o-", label=sutun)
ax.plot([0, 1], [0, 1], "--", color="grey")
ax.set_xlabel("ortalama tahmin edilen risk")
ax.set_ylabel("gözlenen MACE oranı")
ax.legend(fontsize=7)
plt.show()
cevap = round(float(kalibrasyon(y, veri["olasılık, logitlerin ortalaması"].to_numpy())[1]), 2)
teach.check_step("Adım 2.4", "cevap", globals())

**Sonuç ne gösteriyor?** Logitlerin ortalamasının kalibrasyon eğimi 2,78 düzeyindedir: Olasılıkları ortaya yığılır; bu yüzden hastaları erken füzyon kadar iyi sıralamasına karşın düşük riskleri olduğundan yüksek, yüksek riskleri olduğundan düşük tahmin eder.

**2. durak tamamlandı.** [Ders sayfasına dönün](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-08/ders.html#another-hospital-drift-and-recalibration) ve okumaya «Başka bir hastane: Kayma ve yeniden kalibrasyon» bölümünden devam edin. Bir sonraki uygulama durağında ders sayfası sizi yine bu Jupyter Notebook'a getirir.

## Durak 3: Triyaj: Başka bir hastane, bir taburculuk eşiği ve net yarar

*Ders: [3. uygulama durağı](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-08/ders.html#stop-3)* · *Python yolunda aynı sonuç: Kısım 2, 9-10. bölümler*

Hastanesinden çıkan bir model, kullanıldığı yerde denetlenmelidir. İlk adım modeli B hastanesinde ilk hastalarıyla günceller. İkinci adım doğrulama başvurularında bir taburculuk eşiği seçer ve bunu test başvurularında HEART 0-3 ile karşılaştırır. Son adım, yatırma kararı için modelin ve HEART skorunun net yararını sorar.

**Durağın sonunda:** B hastanesinin ilk 300 hastasından öğrendiği logit kaymasını (0,37), modelin eşiğiyle eve gönderilen test hastalarındaki MACE sayısını (9) ve modelin yüzde 20 eşik olasılığındaki net yararını (0,125) bulacaksınız.

In [ ]:
#@title 3. durağın hazırlığı: Bu hücreyi bir kez çalıştırın (durağın verilerini yükler)
if "chest" not in globals():
    chest = pd.concat([triage.presentations("A"), triage.presentations("B")], ignore_index=True)
    A = chest[chest["hospital"] == "A"].reset_index(drop=True)
    B = chest[chest["hospital"] == "B"].reset_index(drop=True)
    tr, va, te = [(A["set"] == s).to_numpy() for s in ("training", "validation", "test")]
    y, yB = A["mace30"].to_numpy(), B["mace30"].to_numpy()
if "fit_logistic" not in globals():
    def fit_logistic(X, y, Cs=(0.01, 0.1, 1.0, 10.0), Xv=None, yv=None):
        """Sütunları standartlaştırır ve bir lojistik regresyon kurar; doğrulama kümesi verilirse doğrulama AUC değeri en büyük olan ters düzenlileştirme gücünü tutar."""
        best = None
        for C in (Cs if Xv is not None else (1.0,)):
            model = make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=5000)).fit(X, y)
            auc = roc_auc_score(yv, model.predict_proba(Xv)[:, 1]) if Xv is not None else 0.0
            if best is None or auc > best[1]:
                best = (model, auc)
        return best[0]


    def logit(p):
        """Bir olasılığın log-odds değerini (logit) verir."""
        p = np.clip(p, 1e-6, 1 - 1e-6)
        return np.log(p / (1 - p))


    def bootstrap_auc(y, p, n=2000, seed=2026):
        """Hastalardan çekilen n bootstrap örneğinde AUC: n değerden oluşan bir dizi."""
        rng = np.random.default_rng(seed)
        out = []
        while len(out) < n:
            i = rng.integers(0, len(y), len(y))
            if 0 < y[i].sum() < len(i):
                out.append(roc_auc_score(y[i], p[i]))
        return np.array(out)


    def paired_bootstrap(y, p1, p2, n=2000, seed=2026):
        """Aynı hastalardan çekilen n bootstrap örneğinde p1'in AUC değeri eksi p2'nin AUC değeri: n farktan oluşan bir dizi."""
        rng = np.random.default_rng(seed)
        out = []
        while len(out) < n:
            i = rng.integers(0, len(y), len(y))
            if 0 < y[i].sum() < len(i):
                out.append(roc_auc_score(y[i], p1[i]) - roc_auc_score(y[i], p2[i]))
        return np.array(out)


    def calibration(y, p):
        """Kalibrasyon sabiti (eğim 1'e sabitlenerek bulunur) ve kalibrasyon eğimi."""
        z = logit(p)
        a = 0.0
        for _ in range(50):
            q = 1 / (1 + np.exp(-(a + z)))
            a += (y - q).sum() / (q * (1 - q)).sum()
        slope = LogisticRegression(C=1e6, max_iter=5000).fit(z[:, None], y).coef_[0, 0]
        return a, slope


    def reliability(y, p, bins=10):
        """Eşit büyüklükte gruplarda ortalama tahmin edilen risk ve gözlenen olay oranı: İki dizi."""
        order = np.argsort(p)
        groups = np.array_split(order, bins)
        return np.array([p[g].mean() for g in groups]), np.array([y[g].mean() for g in groups])
_SETS = {"training": "eğitim", "validation": "doğrulama", "test": "test", "external": "B hastanesi"}
# kümeler ve sonuçlar, bir asistanın kodunun üzerine yazmadığı özel adlarla
_tr, _va, _te = [(A["set"] == _s).to_numpy() for _s in ("training", "validation", "test")]
_y, _yB = A["mace30"].to_numpy(), B["mace30"].to_numpy()


def _unit_table(d):
    """Bir hastanenin başvuruları; ekibin geliş anında bildikleri ve 30 gün içindeki MACE ile birlikte."""
    url = d["troponin_url"]
    return pd.DataFrame({"başvuru": d["patient_id"], "küme": d["set"].map(_SETS), "yaş": d["age"],
                         "cinsiyet": d["sex"].map({"M": "erkek", "F": "kadın"}),
                         "sigara": d["smoker"], "diyabet": d["diabetes"], "hipertansiyon": d["hypertension"],
                         "hiperlipidemi": d["hyperlipidaemia"], "obezite": d["obesity"],
                         "aile öyküsü": d["family_history"], "bilinen koroner arter hastalığı": d["known_cad"],
                         "kalp hızı": d["heart_rate"], "sistolik basınç": d["sbp"], "eGFR": d["egfr"],
                         "gelişte troponin": (d["troponin_0h"] / url).round(2),
                         "bir saat sonra troponin": (d["troponin_1h"] / url).round(2),
                         "ekokardiyogram yapıldı": d["echo_done"], "ejeksiyon fraksiyonu": d["lvef"],
                         "duvar hareket bozukluğu": d["rwma"], "HEART": d["heart"], "MACE": d["mace30"]})


def _course_probabilities():
    """Dersin modellerinin A ve B hastanelerinde verdiği MACE olasılıkları: Beş tek modalite, erken füzyon ve logitlerin ortalaması (bir kez hesaplanır, bir ile üç dakika sürer; sürenin çoğu EKG ağı içindir)."""
    if "_prob_cache" in globals():
        return _prob_cache
    tr, va, y = _tr, _va, _y
    CLINICAL = ["age", "male", "smoker", "diabetes", "hypertension", "hyperlipidaemia", "obesity", "family_history", "known_cad",
                "heart_rate", "sbp", "egfr"]


    def clinical(d):
        """Klinik tablo: Yaş, cinsiyet, risk faktörleri, bilinen koroner arter hastalığı, kalp hızı, sistolik basınç ve eGFR."""
        return d.assign(male=(d["sex"] == "M").astype(int))[CLINICAL].astype(float)


    def laboratory(d):
        """Kendi testinin üst referans sınırının katları olarak troponin: Gelişteki ve bir saat sonraki değerin logaritması, bir saatteki değişim ve ikinci değerin olup olmadığı."""
        url = d["troponin_url"]
        t0, t1 = d["troponin_0h"] / url, d["troponin_1h"] / url
        return pd.DataFrame({"log_trop_0h": np.log(t0), "log_trop_1h": np.log(t1.fillna(t0)),
                             "delta_1h": (t1 - t0).fillna(0.0), "has_1h": t1.notna().astype(float)})


    def echo(d):
        """Yatak başı ekokardiyogram: Yapılıp yapılmadığı, ejeksiyon fraksiyonu (yapılmadıysa 60) ve bölgesel duvar hareket bozukluğu (yapılmadıysa 0)."""
        return pd.DataFrame({"echo_done": d["echo_done"].astype(float), "lvef": d["lvef"].fillna(60.0), "rwma": d["rwma"].fillna(0.0)})

    tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=3, sublinear_tf=True).fit(A.loc[tr, "note_tr"])
    svd = TruncatedSVD(n_components=20, random_state=RANDOM_STATE).fit(tfidf.transform(A.loc[tr, "note_tr"]))


    def note(d):
        """Triyaj notu 20 sayı olarak: Kelimelerin ve kelime çiftlerinin TF-IDF değerleri, eğitim notlarında öğrenilen bir tekil değer ayrışımıyla küçültülür."""
        return pd.DataFrame(svd.transform(tfidf.transform(d["note_tr"])), columns=[f"note_{i + 1}" for i in range(20)])

    ecg_A = triage.ecg_array(A)          # 12 derivasyonlu 4.000 EKG, 100 Hz'de 10 saniye: 4000 x 12 x 1000
    ecg_B = triage.ecg_array(B)


    class ECGNet(nn.Module):
        """12 derivasyonlu bir EKG için tek boyutlu evrişimli ağ: Dört blok, her kanalın zaman boyunca ortalaması ve tek bir çıktı."""
        def __init__(self):
            super().__init__()
            block = lambda a, b: nn.Sequential(nn.Conv1d(a, b, 7, padding=3), nn.BatchNorm1d(b), nn.ReLU(), nn.MaxPool1d(2))
            self.features = nn.Sequential(block(12, 32), block(32, 64), block(64, 64), block(64, 64))
            self.head = nn.Linear(64, 1)

        def embed(self, x):
            return self.features(x).mean(dim=2)

        def forward(self, x):
            return self.head(self.embed(x)).squeeze(1)


    def train_ecg(model, X, y, Xv, yv, epochs=12, seed=2026, verbose=True):
        """64 EKG'lik gruplarla, öğrenme hızı 0,001 olan Adam; her EKG zamanda en fazla yarım saniye kaydırılır. Doğrulama AUC değeri en iyi olan epoch'un ağırlıkları saklanır."""
        torch.manual_seed(seed)
        X, y = torch.as_tensor(X), torch.as_tensor(y, dtype=torch.float32)
        optimiser = torch.optim.Adam(model.parameters(), lr=0.001)
        loss_fn = nn.BCEWithLogitsLoss()
        best, state = -1.0, None
        for epoch in range(epochs):
            model.train()
            order = torch.randperm(len(X))
            for i in range(0, len(X), 64):
                b = order[i:i + 64]
                xb = torch.roll(X[b], int(torch.randint(-50, 51, (1,))), dims=2)
                loss = loss_fn(model(xb), y[b])
                optimiser.zero_grad()
                loss.backward()
                optimiser.step()
            a = roc_auc_score(yv, ecg_predict(model, Xv))
            if a > best:
                best, state = a, copy.deepcopy(model.state_dict())
            if verbose:
                print(f"epoch {epoch + 1}: doğrulama AUC {a:.3f}")
        model.load_state_dict(state)
        return best


    @torch.no_grad()
    def ecg_predict(model, X, batch=256):
        model.eval()
        return torch.cat([torch.sigmoid(model(torch.as_tensor(X[i:i + batch]))) for i in range(0, len(X), batch)]).numpy()


    @torch.no_grad()
    def ecg_embed(model, X, batch=256):
        model.eval()
        return torch.cat([model.embed(torch.as_tensor(X[i:i + batch])) for i in range(0, len(X), batch)]).numpy()
    torch.manual_seed(RANDOM_STATE)
    net = ECGNet()
    train_ecg(net, ecg_A[tr], y[tr], ecg_A[va], y[va], verbose=False)
    emb = lambda X: pd.DataFrame(ecg_embed(net, X), columns=[f"ecg_{i + 1}" for i in range(64)])
    out = {}
    for name, d, X in (("A", A, ecg_A), ("B", B, ecg_B)):
        parts = {"klinik": clinical(d), "troponin": laboratory(d), "ekokardiyogram": echo(d), "not": note(d)}
        out[name] = dict(parts=parts, ecg=ecg_predict(net, X), embedding=emb(X))
    p = {"A": {}, "B": {}}
    for k in out["A"]["parts"]:
        m = fit_logistic(out["A"]["parts"][k][tr], y[tr], Xv=out["A"]["parts"][k][va], yv=y[va])
        for h in ("A", "B"):
            p[h][k] = m.predict_proba(out[h]["parts"][k])[:, 1]
    for h in ("A", "B"):
        p[h]["EKG ağı"] = out[h]["ecg"]
    early = lambda h: pd.concat([out[h]["parts"]["klinik"], out[h]["parts"]["troponin"],
                                 out[h]["parts"]["ekokardiyogram"], out[h]["embedding"], out[h]["parts"]["not"]], axis=1)
    m = fit_logistic(early("A")[tr], y[tr], Xv=early("A")[va], yv=y[va])
    for h in ("A", "B"):
        p[h]["early"] = m.predict_proba(early(h))[:, 1]
        z = np.column_stack([logit(p[h][k]) for k in ["klinik", "troponin", "ekokardiyogram", "EKG ağı", "not"]])
        p[h]["mean"] = 1 / (1 + np.exp(-z.mean(axis=1)))
    globals()["_prob_cache"] = p
    return p


print(len(A), "başvuru A hastanesinde,", len(B), "başvuru B hastanesinde")
print("3. durak hazır.")

### Adım 3.1: Başka bir hastane için bir güncelleme

**Amaç.** Bu adımda modelin logitleri, ortalama riski B hastanesinin ilk 300 hastasındaki olay oranına uyacak biçimde kaydırılır ve sonraki 700 hasta denetlenir.

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter Notebook'uma yapıştırıp çalıştıracağım.  
> Bağlam: B hastanesinin kardiyoloğu olarak hastaları ve uygulaması farklı olan A hastanesinden bir model kullanmak istiyorum ve ona güvenmeden önce risklerinin düzeyini B'nin ilk hastalarıyla düzeltmek istiyorum.  
> Veri: Jupyter Notebook'ta "veri" adlı bir tablo vardır. Tabloda sanal kliniğin ikinci hastanesi olan B hastanesinden, yani sentetik bir veri setinden göğüs ağrısıyla yapılmış 1.000 başvuru, geliş sırasıyla bulunur. Her satır bir başvurudur: "sıra" zamandaki yerini, "olasılık" birinci hastane olan A'da öğrenmiş bir modelin 30 gün içinde majör olumsuz kardiyak olay için verdiği olasılığı verir; "MACE" böyle bir olay görüldüyse 1, aksi hâlde 0 olur.  
> Görev: Olasılıkları logite dönüştür. İlk 300 başvuruda, her logite eklendiğinde kaydırılmış olasılıkların ortalamasını bu 300 başvurudaki MACE oranına eşitleyen kaymayı bul (örneğin Newton adımlarıyla). Kaymayı bütün başvuruların logitlerine ekle. Kalan 700 başvuru için güncellemeden önceki ve sonraki ortalama tahmin edilen riski, gözlenen MACE oranını ve önceki ve sonraki AUC değerini bildir.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: Kaymayı, ortalamaları, gözlenen oranı ve iki AUC değerini yazdır; kaymayı iki ondalık basamakla "cevap" adıyla sakla.  
> Test: Kaymayı bulmak için yalnızca ilk 300 başvuru kullanılmalı ve AUC değişmemeli.  

In [ ]:
#@title Adım 3.1 için veri: Bu hücreyi çalıştırın ("veri" tablosunu hazırlar ve gösterir)
_p = _course_probabilities()["B"]   # ilk seferde bir ile üç dakika sürer
veri = pd.DataFrame({"sıra": np.arange(1, len(B) + 1), "olasılık": _p["early"], "MACE": _yB})
globals().pop("cevap", None)
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

In [ ]:
# Adım 3.1 kontrolü
teach.check_step("Adım 3.1", "cevap", globals())

In [ ]:
#@title Adım 3.1 için hazır kod (yalnızca asistanın kodu kontrolden geçmezse çalıştırın)
_p = _course_probabilities()["B"]   # ilk seferde bir ile üç dakika sürer
veri = pd.DataFrame({"sıra": np.arange(1, len(B) + 1), "olasılık": _p["early"], "MACE": _yB})
p = np.clip(veri["olasılık"].to_numpy(), 1e-6, 1 - 1e-6)
z = np.log(p / (1 - p))
y = veri["MACE"].to_numpy()
ilk = veri["sıra"].to_numpy() <= 300
a = 0.0
for _ in range(50):
    q = 1 / (1 + np.exp(-(a + z[ilk])))
    a += (y[ilk] - q).sum() / (q * (1 - q)).sum()
p_yeni = 1 / (1 + np.exp(-(z + a)))
kalan = ~ilk
print("kayma", round(a, 3), "| önce ortalama", round(p[kalan].mean(), 3), "| sonra", round(p_yeni[kalan].mean(), 3),
      "| gözlenen", round(y[kalan].mean(), 3))
print("AUC", round(roc_auc_score(y[kalan], p[kalan]), 3), round(roc_auc_score(y[kalan], p_yeni[kalan]), 3))
cevap = round(float(a), 2)
teach.check_step("Adım 3.1", "cevap", globals())

**Sonuç ne gösteriyor?** B hastanesi 0,37 kayma öğrenir: A'nın modeli, hastaların daha yaşlı olduğu ve anginanın daha girişimsel tedavi edildiği B hastanesinde riski olduğundan düşük tahmin eder. Güncelleme her hastayı logit ölçeğinde aynı adımla yukarı taşır ve AUC değerini değiştirmez.

### Adım 3.2: Bir taburculuk eşiği

**Amaç.** Bu adımda doğrulama başvurularında modelin HEART 0-3 kadar hastayı eve gönderen bir eşiği seçilir ve bu eşiğin eve gönderdiği test hastalarındaki MACE sayılır.

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter Notebook'uma yapıştırıp çalıştıracağım.  
> Bağlam: Eve gönderilen hasta sayısı aynı kalmak koşuluyla, hastaları HEART 0-3 yerine bir modelin riskine göre taburcu etmenin daha az majör olumsuz kardiyak olay kaçırıp kaçırmayacağını öğrenmek istiyorum.  
> Veri: Jupyter Notebook'ta "veri" adlı bir tablo vardır. Tabloda sanal kliniğin bir göğüs ağrısı ünitesinden, yani sentetik bir veri setinden 800 doğrulama ve 800 test başvurusu bulunur. Her satır bir başvurudur: "küme" kümesini ("doğrulama" ya da "test"), "HEART" HEART skorunu (0-10), "olasılık" dersin beş modalite kullanan bir modelinin 30 gün içinde majör olumsuz kardiyak olay için verdiği olasılığı verir; "MACE" böyle bir olay görüldüyse 1, aksi hâlde 0 olur.  
> Görev: Doğrulama başvurularında HEART skoru 0-3 olan hastaları say; bu sayıya k de. Doğrulama başvurularını olasılığa göre sırala; eşik, en düşük k. olasılıktır. Test başvurularında eşikte ya da altında kalan her hastayı eve gönder; eve gönderilenlerin sayısını, bunlardaki MACE sayısını, yüzde 95 Wilson aralığıyla birlikte oranlarını ve MACE için duyarlılığı bildir. Aynısını test başvurularında HEART 0-3 için de bildir.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: Eşiği ve iki sonucu yazdır; modelin eve gönderdiği test hastalarındaki MACE sayısını "cevap" adıyla sakla.  
> Test: Eşik yalnızca doğrulama başvurularından gelmeli ve modelin eve gönderdiği hasta sayısı HEART skorununkine yakın olmalı.  

In [ ]:
#@title Adım 3.2 için veri: Bu hücreyi çalıştırın ("veri" tablosunu hazırlar ve gösterir)
_p = _course_probabilities()["A"]   # ilk seferde bir ile üç dakika sürer
_k = _va | _te
veri = pd.DataFrame({"küme": np.where(_va, "doğrulama", "test")[_k], "HEART": A["heart"].to_numpy()[_k],
                     "olasılık": _p["early"][_k], "MACE": _y[_k]})
globals().pop("cevap", None)
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

In [ ]:
# Adım 3.2 kontrolü
teach.check_step("Adım 3.2", "cevap", globals())

In [ ]:
#@title Adım 3.2 için hazır kod (yalnızca asistanın kodu kontrolden geçmezse çalıştırın)
_p = _course_probabilities()["A"]   # ilk seferde bir ile üç dakika sürer
_k = _va | _te
veri = pd.DataFrame({"küme": np.where(_va, "doğrulama", "test")[_k], "HEART": A["heart"].to_numpy()[_k],
                     "olasılık": _p["early"][_k], "MACE": _y[_k]})


def wilson(k, n, z=1.96):
    p = k / n
    c = (p + z ** 2 / (2 * n)) / (1 + z ** 2 / n)
    h = z * np.sqrt(p * (1 - p) / n + z ** 2 / (4 * n ** 2)) / (1 + z ** 2 / n)
    return c - h, c + h


do = veri[veri["küme"] == "doğrulama"]
te = veri[veri["küme"] == "test"]
k = int((do["HEART"] <= 3).sum())
esik = float(np.sort(do["olasılık"].to_numpy())[k - 1])
print("eşik", round(esik, 4))
for ad, ev in (("model", te["olasılık"] <= esik), ("HEART 0-3", te["HEART"] <= 3)):
    n_ev, kacan = int(ev.sum()), int(te["MACE"][ev].sum())
    alt, ust = wilson(kacan, n_ev)
    print(ad, "| eve giden", n_ev, "| MACE", kacan, "|", round(kacan / n_ev, 3), (round(alt, 3), round(ust, 3)),
          "| duyarlılık", round(1 - kacan / te["MACE"].sum(), 3))
cevap = int(te["MACE"][te["olasılık"] <= esik].sum())
teach.check_step("Adım 3.2", "cevap", globals())

**Sonuç ne gösteriyor?** Eve gönderilen oran aynıyken model test hastalarında 9 MACE kaçırır; bu sayı HEART 0-3'ün kaçırdığına yakındır. Bu ünitede taburculuk kararı için füzyon skordan daha iyi değildir; 800 test hastası da küçük bir farkı gösteremezdi.

### Adım 3.3: Yatırma kararı için net yarar

**Amaç.** Bu adımda modelin riskine göre ve HEART 4 ya da üstüne göre yatırmanın net yararı yüzde 20 eşik olasılığında hesaplanır.

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter Notebook'uma yapıştırıp çalıştıracağım.  
> Bağlam: Yüzde 20 MACE riski olan bir hastayı, yakalanan her olay için dört gereksiz yatışı kabul ederek yatırırdım; bir modelin olasılıklarının bu karara "HEART skoru 4 ya da üstü olan her hastayı yatır" kuralından daha iyi hizmet edip etmediğini öğrenmek istiyorum.  
> Veri: Jupyter Notebook'ta "veri" adlı bir tablo vardır. Tabloda sanal kliniğin bir göğüs ağrısı ünitesinden, yani sentetik bir veri setinden 800 test başvurusu bulunur. Her satır bir başvurudur: "HEART" HEART skorunu (0-10), "olasılık" dersin beş modalite kullanan bir modelinin 30 gün içinde majör olumsuz kardiyak olay için verdiği olasılığı verir; "MACE" böyle bir olay görüldüyse 1, aksi hâlde 0 olur.  
> Görev: t = 0,20 eşik olasılığında doğru pozitifleri (yatırılanlarda MACE) ve yanlış pozitifleri (yatırılanlarda MACE yok) bir kez olasılığı en az t olan her hastayı yatırmak, bir kez de HEART skoru 4 ya da üstü olan her hastayı yatırmak için say. Net yarar, doğru pozitiflerin hasta sayısına bölümünden yanlış pozitiflerin hasta sayısına bölümü ile t / (1 − t) çarpımının çıkarılmasıdır. Herkesi yatırmanın net yararını da ver.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: Üç stratejinin sayılarını ve net yararını yazdır; modelin net yararını üç ondalık basamakla "cevap" adıyla sakla.  
> Test: Hiç kimseyi yatırmamanın net yararı 0'dır; modelinki 800 test başvurusunun tamamında hesaplanmalı.  

In [ ]:
#@title Adım 3.3 için veri: Bu hücreyi çalıştırın ("veri" tablosunu hazırlar ve gösterir)
_p = _course_probabilities()["A"]   # ilk seferde bir ile üç dakika sürer
veri = pd.DataFrame({"HEART": A["heart"].to_numpy()[_te], "olasılık": _p["early"][_te], "MACE": _y[_te]})
globals().pop("cevap", None)
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

In [ ]:
# Adım 3.3 kontrolü
teach.check_step("Adım 3.3", "cevap", globals())

In [ ]:
#@title Adım 3.3 için hazır kod (yalnızca asistanın kodu kontrolden geçmezse çalıştırın)
_p = _course_probabilities()["A"]   # ilk seferde bir ile üç dakika sürer
veri = pd.DataFrame({"HEART": A["heart"].to_numpy()[_te], "olasılık": _p["early"][_te], "MACE": _y[_te]})
t = 0.20
y = veri["MACE"].to_numpy()
n = len(y)
cevap = None
for ad, yatir in (("model", veri["olasılık"].to_numpy() >= t), ("HEART ≥ 4", veri["HEART"].to_numpy() >= 4),
                         ("herkes", np.ones(n, dtype=bool))):
    dp, yp = int((yatir & (y == 1)).sum()), int((yatir & (y == 0)).sum())
    ny = dp / n - yp / n * t / (1 - t)
    print(ad, "| TP", dp, "| FP", yp, "| net yarar", round(ny, 3))
    if cevap is None:
        cevap = round(float(ny), 3)
teach.check_step("Adım 3.3", "cevap", globals())

**Sonuç ne gösteriyor?** Yüzde 20 eşikte model 0,125 net yarar verir; bu değer, olay görülmeyen çok sayıda hastayı yatıran HEART 4 ya da üstü kuralının açıkça üzerindedir. Yatırma kararı için olasılıklar üç puan grubundan daha iyi hizmet eder.

**3. durak tamamlandı.** [Ders sayfasına dönün](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-08/ders.html#ai-pair-programming-part-8-reviewing-a-fusion-pipeline) ve okumaya «YZ asistanıyla çalışmak, 8. bölüm: Bir füzyon işlem hattını denetlemek» bölümünden devam edin. Haftalık proje prompt yoluyla da yapılabilir: Promptları aşağıdaki «Prompt yolunda haftalık proje» bölümündedir.

## Prompt yolunda haftalık proje

*Haftanın projesi: [8. haftanın README sayfası, Haftalık proje](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/tr/hafta-08/README.md#haftal%C4%B1k-proje)*

Haftalık proje, her adım için bir prompt olmak üzere aşağıdaki promptlarla da yapılabilir. Önce hazırlık hücresini bir kez çalıştırın; hücre projenin verilerini yükler. Ardından her adım, duraklardaki adımlar gibi ilerler. Tek fark şudur: Projede kontrol hücresi ve hazır kod yoktur, çünkü sonuçlar rapora girer. Bir promptun testi sağlanmadan sonraki prompta geçilmez ve her prompt YZ kullanım günlüğüne yazılır.

In [ ]:
#@title Projenin hazırlığı: Bu hücreyi bir kez çalıştırın (projenin verilerini yükler)
if "chest" not in globals():
    chest = pd.concat([triage.presentations("A"), triage.presentations("B")], ignore_index=True)
    A = chest[chest["hospital"] == "A"].reset_index(drop=True)
    B = chest[chest["hospital"] == "B"].reset_index(drop=True)
    tr, va, te = [(A["set"] == s).to_numpy() for s in ("training", "validation", "test")]
    y, yB = A["mace30"].to_numpy(), B["mace30"].to_numpy()
if "fit_logistic" not in globals():
    def fit_logistic(X, y, Cs=(0.01, 0.1, 1.0, 10.0), Xv=None, yv=None):
        """Sütunları standartlaştırır ve bir lojistik regresyon kurar; doğrulama kümesi verilirse doğrulama AUC değeri en büyük olan ters düzenlileştirme gücünü tutar."""
        best = None
        for C in (Cs if Xv is not None else (1.0,)):
            model = make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=5000)).fit(X, y)
            auc = roc_auc_score(yv, model.predict_proba(Xv)[:, 1]) if Xv is not None else 0.0
            if best is None or auc > best[1]:
                best = (model, auc)
        return best[0]


    def logit(p):
        """Bir olasılığın log-odds değerini (logit) verir."""
        p = np.clip(p, 1e-6, 1 - 1e-6)
        return np.log(p / (1 - p))


    def bootstrap_auc(y, p, n=2000, seed=2026):
        """Hastalardan çekilen n bootstrap örneğinde AUC: n değerden oluşan bir dizi."""
        rng = np.random.default_rng(seed)
        out = []
        while len(out) < n:
            i = rng.integers(0, len(y), len(y))
            if 0 < y[i].sum() < len(i):
                out.append(roc_auc_score(y[i], p[i]))
        return np.array(out)


    def paired_bootstrap(y, p1, p2, n=2000, seed=2026):
        """Aynı hastalardan çekilen n bootstrap örneğinde p1'in AUC değeri eksi p2'nin AUC değeri: n farktan oluşan bir dizi."""
        rng = np.random.default_rng(seed)
        out = []
        while len(out) < n:
            i = rng.integers(0, len(y), len(y))
            if 0 < y[i].sum() < len(i):
                out.append(roc_auc_score(y[i], p1[i]) - roc_auc_score(y[i], p2[i]))
        return np.array(out)


    def calibration(y, p):
        """Kalibrasyon sabiti (eğim 1'e sabitlenerek bulunur) ve kalibrasyon eğimi."""
        z = logit(p)
        a = 0.0
        for _ in range(50):
            q = 1 / (1 + np.exp(-(a + z)))
            a += (y - q).sum() / (q * (1 - q)).sum()
        slope = LogisticRegression(C=1e6, max_iter=5000).fit(z[:, None], y).coef_[0, 0]
        return a, slope


    def reliability(y, p, bins=10):
        """Eşit büyüklükte gruplarda ortalama tahmin edilen risk ve gözlenen olay oranı: İki dizi."""
        order = np.argsort(p)
        groups = np.array_split(order, bins)
        return np.array([p[g].mean() for g in groups]), np.array([y[g].mean() for g in groups])
_SETS = {"training": "eğitim", "validation": "doğrulama", "test": "test", "external": "B hastanesi"}
# kümeler ve sonuçlar, bir asistanın kodunun üzerine yazmadığı özel adlarla
_tr, _va, _te = [(A["set"] == _s).to_numpy() for _s in ("training", "validation", "test")]
_y, _yB = A["mace30"].to_numpy(), B["mace30"].to_numpy()


def _unit_table(d):
    """Bir hastanenin başvuruları; ekibin geliş anında bildikleri ve 30 gün içindeki MACE ile birlikte."""
    url = d["troponin_url"]
    return pd.DataFrame({"başvuru": d["patient_id"], "küme": d["set"].map(_SETS), "yaş": d["age"],
                         "cinsiyet": d["sex"].map({"M": "erkek", "F": "kadın"}),
                         "sigara": d["smoker"], "diyabet": d["diabetes"], "hipertansiyon": d["hypertension"],
                         "hiperlipidemi": d["hyperlipidaemia"], "obezite": d["obesity"],
                         "aile öyküsü": d["family_history"], "bilinen koroner arter hastalığı": d["known_cad"],
                         "kalp hızı": d["heart_rate"], "sistolik basınç": d["sbp"], "eGFR": d["egfr"],
                         "gelişte troponin": (d["troponin_0h"] / url).round(2),
                         "bir saat sonra troponin": (d["troponin_1h"] / url).round(2),
                         "ekokardiyogram yapıldı": d["echo_done"], "ejeksiyon fraksiyonu": d["lvef"],
                         "duvar hareket bozukluğu": d["rwma"], "HEART": d["heart"], "MACE": d["mace30"]})


def _course_probabilities():
    """Dersin modellerinin A ve B hastanelerinde verdiği MACE olasılıkları: Beş tek modalite, erken füzyon ve logitlerin ortalaması (bir kez hesaplanır, bir ile üç dakika sürer; sürenin çoğu EKG ağı içindir)."""
    if "_prob_cache" in globals():
        return _prob_cache
    tr, va, y = _tr, _va, _y
    CLINICAL = ["age", "male", "smoker", "diabetes", "hypertension", "hyperlipidaemia", "obesity", "family_history", "known_cad",
                "heart_rate", "sbp", "egfr"]


    def clinical(d):
        """Klinik tablo: Yaş, cinsiyet, risk faktörleri, bilinen koroner arter hastalığı, kalp hızı, sistolik basınç ve eGFR."""
        return d.assign(male=(d["sex"] == "M").astype(int))[CLINICAL].astype(float)


    def laboratory(d):
        """Kendi testinin üst referans sınırının katları olarak troponin: Gelişteki ve bir saat sonraki değerin logaritması, bir saatteki değişim ve ikinci değerin olup olmadığı."""
        url = d["troponin_url"]
        t0, t1 = d["troponin_0h"] / url, d["troponin_1h"] / url
        return pd.DataFrame({"log_trop_0h": np.log(t0), "log_trop_1h": np.log(t1.fillna(t0)),
                             "delta_1h": (t1 - t0).fillna(0.0), "has_1h": t1.notna().astype(float)})


    def echo(d):
        """Yatak başı ekokardiyogram: Yapılıp yapılmadığı, ejeksiyon fraksiyonu (yapılmadıysa 60) ve bölgesel duvar hareket bozukluğu (yapılmadıysa 0)."""
        return pd.DataFrame({"echo_done": d["echo_done"].astype(float), "lvef": d["lvef"].fillna(60.0), "rwma": d["rwma"].fillna(0.0)})

    tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=3, sublinear_tf=True).fit(A.loc[tr, "note_tr"])
    svd = TruncatedSVD(n_components=20, random_state=RANDOM_STATE).fit(tfidf.transform(A.loc[tr, "note_tr"]))


    def note(d):
        """Triyaj notu 20 sayı olarak: Kelimelerin ve kelime çiftlerinin TF-IDF değerleri, eğitim notlarında öğrenilen bir tekil değer ayrışımıyla küçültülür."""
        return pd.DataFrame(svd.transform(tfidf.transform(d["note_tr"])), columns=[f"note_{i + 1}" for i in range(20)])

    ecg_A = triage.ecg_array(A)          # 12 derivasyonlu 4.000 EKG, 100 Hz'de 10 saniye: 4000 x 12 x 1000
    ecg_B = triage.ecg_array(B)


    class ECGNet(nn.Module):
        """12 derivasyonlu bir EKG için tek boyutlu evrişimli ağ: Dört blok, her kanalın zaman boyunca ortalaması ve tek bir çıktı."""
        def __init__(self):
            super().__init__()
            block = lambda a, b: nn.Sequential(nn.Conv1d(a, b, 7, padding=3), nn.BatchNorm1d(b), nn.ReLU(), nn.MaxPool1d(2))
            self.features = nn.Sequential(block(12, 32), block(32, 64), block(64, 64), block(64, 64))
            self.head = nn.Linear(64, 1)

        def embed(self, x):
            return self.features(x).mean(dim=2)

        def forward(self, x):
            return self.head(self.embed(x)).squeeze(1)


    def train_ecg(model, X, y, Xv, yv, epochs=12, seed=2026, verbose=True):
        """64 EKG'lik gruplarla, öğrenme hızı 0,001 olan Adam; her EKG zamanda en fazla yarım saniye kaydırılır. Doğrulama AUC değeri en iyi olan epoch'un ağırlıkları saklanır."""
        torch.manual_seed(seed)
        X, y = torch.as_tensor(X), torch.as_tensor(y, dtype=torch.float32)
        optimiser = torch.optim.Adam(model.parameters(), lr=0.001)
        loss_fn = nn.BCEWithLogitsLoss()
        best, state = -1.0, None
        for epoch in range(epochs):
            model.train()
            order = torch.randperm(len(X))
            for i in range(0, len(X), 64):
                b = order[i:i + 64]
                xb = torch.roll(X[b], int(torch.randint(-50, 51, (1,))), dims=2)
                loss = loss_fn(model(xb), y[b])
                optimiser.zero_grad()
                loss.backward()
                optimiser.step()
            a = roc_auc_score(yv, ecg_predict(model, Xv))
            if a > best:
                best, state = a, copy.deepcopy(model.state_dict())
            if verbose:
                print(f"epoch {epoch + 1}: doğrulama AUC {a:.3f}")
        model.load_state_dict(state)
        return best


    @torch.no_grad()
    def ecg_predict(model, X, batch=256):
        model.eval()
        return torch.cat([torch.sigmoid(model(torch.as_tensor(X[i:i + batch]))) for i in range(0, len(X), batch)]).numpy()


    @torch.no_grad()
    def ecg_embed(model, X, batch=256):
        model.eval()
        return torch.cat([model.embed(torch.as_tensor(X[i:i + batch])) for i in range(0, len(X), batch)]).numpy()
    torch.manual_seed(RANDOM_STATE)
    net = ECGNet()
    train_ecg(net, ecg_A[tr], y[tr], ecg_A[va], y[va], verbose=False)
    emb = lambda X: pd.DataFrame(ecg_embed(net, X), columns=[f"ecg_{i + 1}" for i in range(64)])
    out = {}
    for name, d, X in (("A", A, ecg_A), ("B", B, ecg_B)):
        parts = {"klinik": clinical(d), "troponin": laboratory(d), "ekokardiyogram": echo(d), "not": note(d)}
        out[name] = dict(parts=parts, ecg=ecg_predict(net, X), embedding=emb(X))
    p = {"A": {}, "B": {}}
    for k in out["A"]["parts"]:
        m = fit_logistic(out["A"]["parts"][k][tr], y[tr], Xv=out["A"]["parts"][k][va], yv=y[va])
        for h in ("A", "B"):
            p[h][k] = m.predict_proba(out[h]["parts"][k])[:, 1]
    for h in ("A", "B"):
        p[h]["EKG ağı"] = out[h]["ecg"]
    early = lambda h: pd.concat([out[h]["parts"]["klinik"], out[h]["parts"]["troponin"],
                                 out[h]["parts"]["ekokardiyogram"], out[h]["embedding"], out[h]["parts"]["not"]], axis=1)
    m = fit_logistic(early("A")[tr], y[tr], Xv=early("A")[va], yv=y[va])
    for h in ("A", "B"):
        p[h]["early"] = m.predict_proba(early(h))[:, 1]
        z = np.column_stack([logit(p[h][k]) for k in ["klinik", "troponin", "ekokardiyogram", "EKG ağı", "not"]])
        p[h]["mean"] = 1 / (1 + np.exp(-z.mean(axis=1)))
    globals()["_prob_cache"] = p
    return p


print(len(A), "başvuru A hastanesinde,", len(B), "başvuru B hastanesinde")



def _project_unit():
    """İlk adım için A hastanesinin başvuruları: Ünite tablosu ile dersin EKG ve not modellerinin puanları."""
    p = _course_probabilities()["A"]
    t = _unit_table(A)
    t["EKG puanı"] = logit(p["EKG ağı"]).round(3)
    t["not puanı"] = logit(p["not"]).round(3)
    return t


def _project_both():
    """A hastanesinin doğrulama ve test başvuruları ile B hastanesinin başvuruları; dersin modelinin olasılığı, HEART skoru ve MACE ile birlikte."""
    p = _course_probabilities()
    k = _va | _te
    a = pd.DataFrame({"küme": np.where(_va, "doğrulama", "test")[k], "sıra": 0, "HEART": A["heart"].to_numpy()[k],
                      "olasılık": p["A"]["early"][k], "MACE": _y[k]})
    b = pd.DataFrame({"küme": "B hastanesi", "sıra": np.arange(1, len(B) + 1), "HEART": B["heart"].to_numpy(),
                      "olasılık": p["B"]["early"], "MACE": _yB})
    return pd.concat([a, b], ignore_index=True)
print("Proje hazır.")

### Proje adımı 1: Tek modaliteler, füzyon ve HEART

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter Notebook'uma yapıştırıp çalıştıracağım.  
> Bağlam: Bir göğüs ağrısı ünitesinin her modalitesinden ve hepsinden birlikte kurulan 30 gün içindeki MACE modellerini HEART skoruyla, her farktan ne kadar emin olunabileceğini gösteren aralıklarla karşılaştırmak istiyorum.  
> Veri: Jupyter Notebook'ta "veri" adlı bir tablo vardır. Tabloda sanal kliniğin bir göğüs ağrısı ünitesinden, yani sentetik bir veri setinden göğüs ağrısıyla yapılmış 4.000 başvuru bulunur. Her satır bir başvurudur: "başvuru" numarasını, "küme" kümesini verir ("eğitim", 2.400; "doğrulama", 800; "test", 800). Klinik tablo; "yaş" (yıl), "cinsiyet" ("erkek" ya da "kadın"), varsa 1 olan yedi risk faktörü ("sigara", "diyabet", "hipertansiyon", "hiperlipidemi", "obezite", "aile öyküsü" ve "bilinen koroner arter hastalığı"), "kalp hızı" (dakikada), "sistolik basınç" (mmHg) ve "eGFR" sütunlarını verir. Yüksek duyarlıklı troponin T, testin üst referans sınırının katları olarak verilir: "gelişte troponin" ve ölçülmediğinde boş kalan "bir saat sonra troponin". Yatak başı ekokardiyogram "ekokardiyogram yapıldı" (1 ya da 0) sütununu ve yapıldıysa "ejeksiyon fraksiyonu" (%) ile "duvar hareket bozukluğu" (bölgesel bir bozukluk görüldüyse 1) sütunlarını verir; yapılmadıysa bu iki sütun boştur. "HEART" HEART skorudur; "MACE" ise 30 gün içinde majör olumsuz kardiyak olay görüldüyse 1, aksi hâlde 0 olur. Dersin eğitim başvurularıyla eğitilmiş modellerinden iki sütun daha gelir: "EKG puanı", bir ağın 12 derivasyonlu EKG'den verdiği MACE olasılığının logiti; "not puanı" ise triyaj notundan gelen olasılığın logitidir.  
> Görev: Beş öznitelik grubu oluştur: Klinik tablo (yaş, erkek için 1 olan cinsiyet, yedi risk faktörü, kalp hızı, sistolik basınç, eGFR); troponin (gelişteki troponinin logaritması, ölçülmediyse gelişteki değerle bir saat sonraki troponinin logaritması, ölçülmediyse 0 ile bir saatteki değişim ve ikinci troponin ölçüldüyse 1 olan bir sütun); ekokardiyogram (yapılıp yapılmadığı, yapılmadıysa 60 ile ejeksiyon fraksiyonu ve 0 ile duvar hareket bozukluğu); EKG puanı ve not puanı. Her grup için tek başına ve bütün gruplar için birlikte (erken füzyon) birer model kur; her birini şöyle kur. Sütunları eğitim başvurularıyla standartlaştır ve eğitim başvurularında bir lojistik regresyon kur. Düzenlileştirmenin miktarını doğrulama başvurularındaki ROC eğrisi altındaki alana (AUC) göre seç; gücünün tersi için 0,01, 0,1, 1 ve 10 değerlerini dene ve seçilen modelin test başvurularındaki AUC değerini hesapla. Her AUC değerine test başvurularından çekilen 2.000 bootstrap örneğinden, 2,5. ve 97,5. yüzdelikler arasında bir yüzde 95 aralığı ver; her modelin HEART skoruna göre farkını aynı örneklerde hesapla. Sonucun tekrarlanabilmesi için seed değerini 2026 olarak sabitle.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: Her model ve HEART için bir satırdan oluşan bir tablo göster: Aralığıyla birlikte test AUC değeri ve aralığıyla birlikte HEART skoruna göre farkı; erken füzyonun test AUC değerini üç ondalık basamakla "cevap" adıyla sakla.  
> Test: Hiçbir seçim için test başvurusu kullanılmamalı ve HEART skoru hiçbir modelin özniteliği olmamalı.  

In [ ]:
#@title Proje adımı 1 için veri: Bu hücreyi çalıştırın ("veri" tablosunu hazırlar ve gösterir)
veri = _project_unit()   # ilk seferde bir ile üç dakika sürer
globals().pop("cevap", None)
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

### Proje adımı 2: İki hastanede kalibrasyon

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter Notebook'uma yapıştırıp çalıştıracağım.  
> Bağlam: MACE için bir risk modelinin olasılıklarına, öğrendiği hastanenin test hastalarında ve ikinci bir hastanede, o hastanenin ilk hastalarıyla yapılan bir güncellemeden önce ve sonra güvenilip güvenilemeyeceğini öğrenmek istiyorum.  
> Veri: Jupyter Notebook'ta "veri" adlı bir tablo vardır. Tabloda sanal kliniğin iki hastanesinden, yani sentetik bir veri setinden göğüs ağrısıyla yapılmış başvurular bulunur: A hastanesinin 800 doğrulama ve 800 test başvurusu ile B hastanesinin 1.000 başvurusu. Her satır bir başvurudur: "küme" "doğrulama", "test" ya da "B hastanesi" değerini, "sıra" B'deki bir başvurunun zamandaki yerini (1-1.000; A için 0), "HEART" HEART skorunu (0-10), "olasılık" A hastanesinin beş modaliteyi birleştiren bir modelinin 30 gün içinde majör olumsuz kardiyak olay için verdiği olasılığı verir; "MACE" böyle bir olay görüldüyse 1, aksi hâlde 0 olur.  
> Görev: A'nın test başvuruları için ortalama tahmin edilen riski MACE oranının yanında hesapla. Kalibrasyon sabitini (eğim 1'e sabitlenmişken ortalama tahmin edilen riski gözlenen orana eşitleyen logit kayması), kalibrasyon eğimini (MACE'in olasılığın logiti üzerine düzenlileştirmesiz bir lojistik regresyonu) ve Brier skorunu hesapla. Köşegenle birlikte, eşit büyüklükte on gruplu bir güvenilirlik eğrisi çiz. B hastanesi için sırası 1-300 olan başvurularda, ortalama tahmin edilen risklerini MACE oranlarına eşitleyen logit kaymasını bul, bunu B'nin bütün başvurularının logitlerine ekle ve ölçütleri ve eğriyi, sırası 301-1.000 olan başvurular için güncellemeden önce ve sonra tekrarla.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: Ölçütleri üç satırlık bir tabloda (A test, B önce, B sonra) ve üç eğriyi göster; güncellemeden sonra B'deki kalibrasyon sabitini iki ondalık basamakla "cevap" adıyla sakla.  
> Test: Kaymayı bulmak için B'nin yalnızca sırası 1-300 olan başvuruları kullanılmalı.  

In [ ]:
#@title Proje adımı 2 için veri: Bu hücreyi çalıştırın ("veri" tablosunu hazırlar ve gösterir)
veri = _project_both()   # ilk seferde bir ile üç dakika sürer
globals().pop("cevap", None)
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

### Proje adımı 3: İki taburculuk eşiği

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter Notebook'uma yapıştırıp çalıştıracağım.  
> Bağlam: Göğüs ağrılı hastaları bir modelin riskine göre taburcu etmek için bir eşik seçmek ve bu eşiğin, seçildiği hastanede ve başka bir hastanede, HEART 0-3'ün yanında kaç olay kaçıracağını öğrenmek istiyorum.  
> Veri: Jupyter Notebook'ta "veri" adlı bir tablo vardır. Tabloda sanal kliniğin iki hastanesinden, yani sentetik bir veri setinden göğüs ağrısıyla yapılmış başvurular bulunur: A hastanesinin 800 doğrulama ve 800 test başvurusu ile B hastanesinin 1.000 başvurusu. Her satır bir başvurudur: "küme" "doğrulama", "test" ya da "B hastanesi" değerini, "sıra" B'deki bir başvurunun zamandaki yerini (1-1.000; A için 0), "HEART" HEART skorunu (0-10), "olasılık" A hastanesinin beş modaliteyi birleştiren bir modelinin 30 gün içinde majör olumsuz kardiyak olay için verdiği olasılığı verir; "MACE" böyle bir olay görüldüyse 1, aksi hâlde 0 olur.  
> Görev: Yalnızca A'nın doğrulama başvurularında iki eşik seç. Birincisi: HEART skoru 0-3 olan doğrulama başvurularını say (k) ve en düşük k. olasılığı al. İkincisi: MACE görülen doğrulama başvurularının en fazla yüzde 2'sinin eşikte ya da altında kaldığı en büyük olasılık. Her eşik ve HEART 0-3 için A'nın test başvurularında ve B'nin sırası 301-1.000 olan başvurularında şunları bildir: Eve gönderilenlerin (eşikte ya da altında kalanların) sayısı ve oranı, yüzde 95 Wilson aralığıyla birlikte bunlardaki MACE ve MACE için duyarlılık.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: Her hastanedeki her kural için bir satırdan oluşan bir tablo göster; birinci eşikte eve gönderilen A test başvurularındaki MACE sayısını "cevap" adıyla sakla.  
> Test: İki eşik de yalnızca A'nın doğrulama başvurularından gelmeli.  

In [ ]:
#@title Proje adımı 3 için veri: Bu hücreyi çalıştırın ("veri" tablosunu hazırlar ve gösterir)
veri = _project_both()   # ilk seferde bir ile üç dakika sürer
globals().pop("cevap", None)
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

### Proje adımı 4: Kalp ekibi için bir karar eğrisi

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter Notebook'uma yapıştırıp çalıştıracağım.  
> Bağlam: Bir MACE risk modelinin hangi karar için yardımcı olduğunu bir kalp ekibine göstermek istiyorum; bunun için modelin net yararını bir dizi yatırma eşiğinde HEART skorununkiyle karşılaştıracağım.  
> Veri: Jupyter Notebook'ta "veri" adlı bir tablo vardır. Tabloda sanal kliniğin iki hastanesinden, yani sentetik bir veri setinden göğüs ağrısıyla yapılmış başvurular bulunur: A hastanesinin 800 doğrulama ve 800 test başvurusu ile B hastanesinin 1.000 başvurusu. Her satır bir başvurudur: "küme" "doğrulama", "test" ya da "B hastanesi" değerini, "sıra" B'deki bir başvurunun zamandaki yerini (1-1.000; A için 0), "HEART" HEART skorunu (0-10), "olasılık" A hastanesinin beş modaliteyi birleştiren bir modelinin 30 gün içinde majör olumsuz kardiyak olay için verdiği olasılığı verir; "MACE" böyle bir olay görüldüyse 1, aksi hâlde 0 olur.  
> Görev: A'nın test başvurularını kullan. 0,01'den 0,30'a 0,01'lik adımlarla her t eşik olasılığı için olasılığı en az t olan her başvuruyu yatırmanın, HEART skoru 4 ya da üstü olan her başvuruyu yatırmanın ve herkesi yatırmanın net yararını hesapla: Doğru pozitiflerin başvuru sayısına bölümünden yanlış pozitiflerin başvuru sayısına bölümü ile t / (1 − t) çarpımının çıkarılması. Hiç kimseyi yatırmamanın 0 çizgisiyle birlikte üç eğriyi çiz.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: Karar eğrisini göster ve 0,05, 0,10, 0,20 ve 0,30 eşiklerinde üç net yararı yazdır; modelin 0,20 eşikteki net yararını üç ondalık basamakla "cevap" adıyla sakla.  
> Test: Hiç kimseyi yatırmamanın net yararı her eşikte 0'dır.  

In [ ]:
#@title Proje adımı 4 için veri: Bu hücreyi çalıştırın ("veri" tablosunu hazırlar ve gösterir)
veri = _project_both()   # ilk seferde bir ile üç dakika sürer
globals().pop("cevap", None)
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

<a id="part-2"></a>

# Kısım 2: Python yolu

**Kimler için?** Bu kısım Python yazmayı öğrenmek isteyen katılımcılar içindir. Kısım 1'i çalışmış olmanız gerekmez: 0. bölümden sonra doğrudan buradan başlayın.

**Nasıl ilerler?** Hücreleri yukarıdan aşağıya Shift+Enter ile çalıştırın. Baştaki *Python adımı*, haftanın gerektirdiği dil bilgisini öğretir. Numaralı bölümlerdeki alıştırmalar hemen *doğru* ya da *henüz değil* yazar; örnek çözümler katlanmış hücrelerde durur ve ancak denedikten sonra açılmalıdır. *Gözlemle ve düşün* hücreleri açık uçlu sorular sorar; bu soruların çözüm hücresi yoktur. *Python notu* hücreleri, yeni bir yapının ilk kullanıldığı yerde dili açıklar. *İleri düzey (isteğe bağlı)* olarak işaretlenen hücreler, programlama bilenler için ek çalışmalardır. Bu kısma ortadan başlayan katılımcı, o bölümün başındaki *Telafi* hücresini çalıştırır. Bazı alıştırma hücreleri `veri` adlı bir tabloyu hazırlayan bir satırla başlar. Bu tablo, etkileşimli labın prompt atölyesindeki alıştırma promptları için hazırlanmıştır; Python yolunda alıştırma, metninde adı geçen tablolarla çözülebilir.

## Python adımı 8: Verileri birleştirmek

Bu hafta bir hastanın verileri birkaç tablodan ve diziden gelir ve doğru satırda buluşmaları gerekir. Bu adımda iki tablonun hasta numarasına göre `merge` ile nasıl birleştirildiği, bir sinyal dizisinin tablosuyla nasıl hizalı kaldığı, tablonun bir maskesinin dizide aynı hastaları nasıl seçtiği ve eksik değerlerin nasıl bulunduğu, doldurulduğu ve işaretlendiği gösterilmektedir. Her örnek sanal kliniğin göğüs ağrısı ünitesini kullanır.

In [ ]:
# Python alıştırması: dili öğrenmek için küçük bir örnek
unit = triage.presentations("A")
clinical_part = unit[["patient_id", "age", "sex", "heart_rate", "sbp"]]
lab_part = unit[["patient_id", "troponin_0h", "troponin_1h", "troponin_url"]].sample(frac=1, random_state=RANDOM_STATE)
print("ilk satırlar:", clinical_part["patient_id"].head(3).tolist(), lab_part["patient_id"].head(3).tolist())
joined = clinical_part.merge(lab_part, on="patient_id", how="left")
print(clinical_part.shape, "+", lab_part.shape, "→", joined.shape)
print(joined[joined["patient_id"] == "CP-3304"])

**Python notu.** `triage.presentations("A")` A hastanesinin 4.000 başvurusunu, her biri bir satır olacak biçimde oluşturur. Tablonun iki bölümü çift köşeli parantezle, yani bir sütun adı listesiyle kesilir; laboratuvar bölümü de başka bir sistemden başka bir sırayla gelmiş gibi `sample(frac=1)` ile karıştırılır. `merge(lab_part, on="patient_id", how="left")`, hasta numaralarının uyuştuğu her yerde ikinci tablonun sütunlarını satır satır birincinin sütunlarının yanına koyar; *left*, ikinci tabloda karşılığı olmasa da birinci tablonun her satırını tutar ve bu durumda değeri boş bırakır. İki tabloyu asla sıralarına göre birleştirmeyin: Bir hastanın değerlerinin buluşmasını yalnızca ortak bir hasta numarası güvenceye alır.

In [ ]:
# Python alıştırması: bir sinyal ve satırı
first = unit.head(5)
signals = triage.ecg_array(first)
print("sinyaller:", signals.shape)
row = 3
print(first["patient_id"].iloc[row], first["ecg_reading"].iloc[row], signals[row].shape)
normal = (first["ecg_reading"] == "normal").to_numpy()
print(normal, "→", signals[normal].shape)

**Python notu.** `triage.ecg_array`, bir tablonun her satırının 12 derivasyonlu EKG'sini sentezler ve 5 × 12 × 1000 boyutlu tek bir dizi döndürür: 5 başvuru, 12 derivasyon ve saniyede 100 örnekle 10 saniye. Dizinin 3. satırı tablonun 3. satırına aittir; çünkü ikisi aynı sırayla oluşturulmuştur ve bu sırada kalmaları gerekir: Diziye aynısını yapmadan tabloyu sıralamak ya da süzmek, bir hastanın EKG'sini başka bir hastaya verirdi. Tablodan `.to_numpy()` ile oluşturulan ve `True` ile `False` değerlerinden oluşan bir maske, ikisinde de aynı hastaları seçer: `first[normal]` ve `signals[normal]`.

In [ ]:
# Python alıştırması: eksik parçalar
print(unit[["troponin_1h", "lvef", "rwma"]].isna().mean().round(3))
filled = unit["lvef"].fillna(60.0)
done = unit["lvef"].notna().astype(int)
print(pd.DataFrame({"lvef": unit["lvef"], "doldurulmuş": filled, "yapıldı": done}).head(6))
print(pd.concat([unit[["age"]], done.rename("echo_done"), filled.rename("lvef")], axis=1).shape)

**Python notu.** `isna()` her boş hücre için `True` verir; `True` ve `False` değerlerinin ortalaması da her sütundaki boş hücrelerin oranıdır. `fillna(60.0)` boşluklara tipik bir ejeksiyon fraksiyonu koyar; `notna().astype(int)` ise bir değerin var olduğu bilgisini kendi sütununda tutar: Ekokardiyogram yapıldıysa 1, yapılmadıysa 0. Böylece model hem doldurulmuş değeri hem de boşluğu görür. `pd.concat([...], axis=1)` aynı satırlara sahip tabloları ve sütunları yan yana koyar; varsayılan değer olan `axis=0` ise onları alt alta koyardı.

**Hızlı kontrol.** `X` tablosunda 4.000 hasta, `Y` tablosunda bunların 3.000'i vardır. `X.merge(Y, on="patient_id", how="left")` kaç satırdan oluşur?

<details><summary>Cevap</summary>

4.000: Sol birleştirme `X` tablosunun her satırını tutar; `Y` tablosunda satırı olmayan 1.000 hasta, bu tablonun sütunlarında boş değer alır.

</details>

**Alıştırma 8.1.** A hastanesindeki başvuruların ne kadarında ikinci troponin ölçülmüştür? Oranı üç ondalık basamakla saklayın.

In [ ]:
veri = alistirma_tablosu("8.1")   # bu alıştırmanın promptundaki tablo (etkileşimli labın F bölümü)
cevap = None   # cevabınız
teach.check("Alıştırma 8.1", cevap)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
cevap = round(float(unit["troponin_1h"].notna().mean()), 3)
teach.check("Alıştırma 8.1", cevap)

**Alıştırma 8.2.** Ekokardiyogramı olan başvurularda (`echo_done` değeri 1) MACE (`mace30`) görülenlerin oranı nedir? Üç ondalık basamakla saklayın.

In [ ]:
veri = alistirma_tablosu("8.2")   # bu alıştırmanın promptundaki tablo (etkileşimli labın F bölümü)
cevap = None   # cevabınız
teach.check("Alıştırma 8.2", cevap)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
cevap = round(float(unit.loc[unit["echo_done"] == 1, "mace30"].mean()), 3)
teach.check("Alıştırma 8.2", cevap)

**Alıştırma 8.3.** A hastanesindeki başvurulardan kaçında hem ikinci troponin hem de ekokardiyogram vardır? Sayıyı saklayın.

In [ ]:
veri = alistirma_tablosu("8.3")   # bu alıştırmanın promptundaki tablo (etkileşimli labın F bölümü)
cevap = None   # cevabınız
teach.check("Alıştırma 8.3", cevap)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
cevap = int((unit["troponin_1h"].notna() & (unit["echo_done"] == 1)).sum())
teach.check("Alıştırma 8.3", cevap)

## 1. Kayıt sistemi ve üç modalite

*Ders: [Mini proje: EKG sinyali + ekokardiyografi + lab değerleri füzyon modeli ile KY risk sınıflandırması](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-08/ders.html#mini-project-heart-failure-risk-classification-with-a-fusion-model-of-the-ecg-signal-echocardiography-and-laboratory-values)*

Mini proje sanal kliniğin kayıt sistemine izlem verileriyle birlikte döner. Sonuç değişkeni 24 ay içinde ölüm ya da kalp yetersizliği nedeniyle hastaneye yatıştır; 2. haftada görüldüğü gibi, olay olmadan 24 aydan kısa izlenen hastalar dışarıda bırakılır. Her hastayı üç modalite tarif eder: EKG, ekokardiyogram ve laboratuvar değerleri. Bu bölüm her modalite için tek başına bir lojistik regresyon kurar.

**Formüllerden koda.** $m$ modalitesinin sayıları, $x_m$: `registry[HF_GROUPS[name]]`. Tek bir modalitenin modeli ve olasılıkları, $p_m$: `fit_logistic(...)` ve `probs_hf[name]`. Düzenlileştirme doğrulama hastalarında seçilir.

**Buradan mı başlıyorsunuz?** Aşağıdaki telafi hücresini bir kez çalıştırın. Hücre, 1. bölümün ihtiyaç duyduğu önceki bölümlerin kodunu, çıktılarını göstermeden çalıştırır.

In [ ]:
#@title Telafi: Jupyter Notebook'a buradan başlıyorsanız bu hücreyi bir kez çalıştırın
def _catch_up(upto, name="NB08_cok_modlu_yapay_zeka_ve_veri_fuzyonu.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-08/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Telafi tamamlandı: Önceki bölümlerin kodu çalıştırıldı.")


_catch_up(1)

In [ ]:
registry = followup.add_followup(cohort.make_cohort(3600))
# 24 ay içinde bir olay; olay olmadan 24 aydan kısa izlenen hastalar dışarıda bırakılır
registry = registry[(registry["event"] == 1) | (registry["followup_months"] >= 24)].reset_index(drop=True)
registry["event24"] = ((registry["event"] == 1) & (registry["followup_months"] <= 24)).astype(int)
number = registry["patient_id"].str[3:].astype(int)
registry["set"] = np.select([number <= 2160, number <= 2880], ["training", "validation"], "test")
registry["lbbb"] = (registry["ecg_conduction"] == "LBBB").astype(int)
registry["af_ecg"] = (registry["ecg_rhythm"] == "AF").astype(int)
registry["log_ntprobnp"] = np.log(registry["ntprobnp"])
registry["log_troponin"] = np.log(registry["troponin"])
HF_GROUPS = {
    "ECG": ["ecg_hr", "ecg_qrs", "ecg_qt", "lbbb", "af_ecg"],
    "eko": ["lvef", "lvedd_mm", "lavi", "e_e_prime", "tapse_mm", "lv_mass_index"],
    "laboratuvar": ["log_ntprobnp", "egfr", "log_troponin"],
}

print(len(registry), "hasta,", int(registry["event24"].sum()), "hastada 24 ay içinde olay")
print(registry.groupby("set")["event24"].agg(["size", "sum"]))
for name, cols in HF_GROUPS.items():
    print(f"{name:14}", cols)

**Python notu.** `followup.add_followup(cohort.make_cohort(3600))`, kayıt sistemindeki 3.600 hastayı 2. haftanın izlem verileriyle birlikte oluşturur: İzlenen ay sayısı, bir olay olup olmadığı ve olduysa hangisi olduğu. `|` içeren satır, olay görülen ya da en az 24 ay izlenen hastaları tutar; ötekiler dışarıda bırakılır, çünkü 24. ayda başlarına ne geldiğini kimse bilmez. `str[3:]` VC-0317 gibi bir hasta numarasından sayıyı keser; `np.select` de bu sayıyı bir kümeye çevirir: 2.160'a kadar eğitim, 2.880'e kadar doğrulama, geri kalanı test. `HF_GROUPS`, bir modalitenin adından sütunlarının listesine giden bir sözlüktür; NT-proBNP ve troponin çok çarpık dağıldıkları için logaritma olarak girer.

In [ ]:
def fit_logistic(X, y, Cs=(0.01, 0.1, 1.0, 10.0), Xv=None, yv=None):
    """Sütunları standartlaştırır ve bir lojistik regresyon kurar; doğrulama kümesi verilirse doğrulama AUC değeri en büyük olan ters düzenlileştirme gücünü tutar."""
    best = None
    for C in (Cs if Xv is not None else (1.0,)):
        model = make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=5000)).fit(X, y)
        auc = roc_auc_score(yv, model.predict_proba(Xv)[:, 1]) if Xv is not None else 0.0
        if best is None or auc > best[1]:
            best = (model, auc)
    return best[0]


def logit(p):
    """Bir olasılığın log-odds değerini (logit) verir."""
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))


def bootstrap_auc(y, p, n=2000, seed=2026):
    """Hastalardan çekilen n bootstrap örneğinde AUC: n değerden oluşan bir dizi."""
    rng = np.random.default_rng(seed)
    out = []
    while len(out) < n:
        i = rng.integers(0, len(y), len(y))
        if 0 < y[i].sum() < len(i):
            out.append(roc_auc_score(y[i], p[i]))
    return np.array(out)


def paired_bootstrap(y, p1, p2, n=2000, seed=2026):
    """Aynı hastalardan çekilen n bootstrap örneğinde p1'in AUC değeri eksi p2'nin AUC değeri: n farktan oluşan bir dizi."""
    rng = np.random.default_rng(seed)
    out = []
    while len(out) < n:
        i = rng.integers(0, len(y), len(y))
        if 0 < y[i].sum() < len(i):
            out.append(roc_auc_score(y[i], p1[i]) - roc_auc_score(y[i], p2[i]))
    return np.array(out)


def calibration(y, p):
    """Kalibrasyon sabiti (eğim 1'e sabitlenerek bulunur) ve kalibrasyon eğimi."""
    z = logit(p)
    a = 0.0
    for _ in range(50):
        q = 1 / (1 + np.exp(-(a + z)))
        a += (y - q).sum() / (q * (1 - q)).sum()
    slope = LogisticRegression(C=1e6, max_iter=5000).fit(z[:, None], y).coef_[0, 0]
    return a, slope


def reliability(y, p, bins=10):
    """Eşit büyüklükte gruplarda ortalama tahmin edilen risk ve gözlenen olay oranı: İki dizi."""
    order = np.argsort(p)
    groups = np.array_split(order, bins)
    return np.array([p[g].mean() for g in groups]), np.array([y[g].mean() for g in groups])

sets_r = registry["set"].to_numpy()
rtr, rva, rte = sets_r == "training", sets_r == "validation", sets_r == "test"
yr = registry["event24"].to_numpy()
probs_hf = {}
for name, cols in HF_GROUPS.items():
    model = fit_logistic(registry.loc[rtr, cols], yr[rtr], Xv=registry.loc[rva, cols], yv=yr[rva])
    probs_hf[name] = model.predict_proba(registry[cols])[:, 1]
    print(f"{name:14} doğrulama AUC {roc_auc_score(yr[rva], probs_hf[name][rva]):.3f}   test AUC {roc_auc_score(yr[rte], probs_hf[name][rte]):.3f}")

**Python notu.** `fit_logistic` sütunları standartlaştırır ve dört düzenlileştirme gücünün (`Cs`) her biri için bir lojistik regresyon kurar; doğrulama hastalarındaki AUC değeri en büyük olan modeli tutar. Test hastaları hiçbir seçim için kullanılmaz. `registry.loc[rtr, cols]` eğitim hastalarının satırlarını ve tek bir modalitenin sütunlarını alır. `predict_proba(...)[:, 1]` kayıt sistemindeki her hasta için bir olay olasılığı verir; böylece aynı dizi üç kümeye de hizmet eder.

**Gözlemle ve düşün.** Kalp yetersizliği olaylarını tek başına en iyi hangi modalite öngörüyor ve NT-proBNP, öteki iki modalitenin yalnızca dolaylı olarak gösterdiği neyi ölçer?

**Alıştırma A.** Kayıt sistemindeki test hastalarından kaçında 24 ay içinde olay görülmüştür? Sayıyı saklayın.

In [ ]:
veri = alistirma_tablosu("A")   # bu alıştırmanın promptundaki tablo (etkileşimli labın F bölümü)
cevap = None   # cevabınız
teach.check("Alıştırma A", cevap)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
cevap = int(yr[rte].sum())
teach.check("Alıştırma A", cevap)

## 2. Erken ve geç füzyon

*Ders: [Mini proje: EKG sinyali + ekokardiyografi + lab değerleri füzyon modeli ile KY risk sınıflandırması](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-08/ders.html#mini-project-heart-failure-risk-classification-with-a-fusion-model-of-the-ecg-signal-echocardiography-and-laboratory-values)*

Erken füzyon, üç modalitenin 14 değerini yan yana koyarak tek bir model kurar. Geç füzyon 1. bölümün üç modelini iki yolla birleştirir. Logitlerin ortalaması eşit bir oylamadır; istifleme ise her modelin ne kadar sayılacağını doğrulama hastalarında ikinci bir lojistik regresyonla öğrenir.

**Formüllerden koda.** $[\,x_1; x_2; x_3\,]$: `registry[cols_all]`. $z_m = \ln(p_m/(1 - p_m))$: `logit(probs_hf[name])`, yani `logits_hf` dizisinin bir sütunu. Ortalama $\frac{1}{M}\sum_m z_m$: `logits_hf.mean(axis=1)`. İstifleme $b + \sum_m w_m z_m$: `stack_hf`; $w_m$ değerleri `stack_hf.coef_`, $b$ ise `stack_hf.intercept_` içindedir.

**Buradan mı başlıyorsunuz?** Aşağıdaki telafi hücresini bir kez çalıştırın. Hücre, 2. bölümün ihtiyaç duyduğu önceki bölümlerin kodunu, çıktılarını göstermeden çalıştırır.

In [ ]:
#@title Telafi: Jupyter Notebook'a buradan başlıyorsanız bu hücreyi bir kez çalıştırın
def _catch_up(upto, name="NB08_cok_modlu_yapay_zeka_ve_veri_fuzyonu.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-08/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Telafi tamamlandı: Önceki bölümlerin kodu çalıştırıldı.")


_catch_up(2)

In [ ]:
cols_all = sum(HF_GROUPS.values(), [])
early_hf = fit_logistic(registry.loc[rtr, cols_all], yr[rtr], Xv=registry.loc[rva, cols_all], yv=yr[rva])
probs_hf["early"] = early_hf.predict_proba(registry[cols_all])[:, 1]
logits_hf = np.column_stack([logit(probs_hf[name]) for name in HF_GROUPS])
probs_hf["late"] = 1 / (1 + np.exp(-logits_hf.mean(axis=1)))
stack_hf = LogisticRegression(max_iter=5000).fit(logits_hf[rva], yr[rva])
probs_hf["stacked"] = stack_hf.predict_proba(logits_hf)[:, 1]
print("istifleme ağırlıkları:", dict(zip(HF_GROUPS, np.round(stack_hf.coef_[0], 2))), "| sabit terim", round(stack_hf.intercept_[0], 2))
auc_hf = {name: roc_auc_score(yr[rte], p[rte]) for name, p in probs_hf.items()}
fig, ax = plt.subplots(figsize=(6.4, 3))
ax.barh(list(auc_hf), list(auc_hf.values()), color=["#8A8A8A"] * 3 + ["#1F5F8B", "#0E7A78", "#6E46A0"])
for i, v in enumerate(auc_hf.values()):
    ax.text(v + 0.005, i, f"{v:.3f}", va="center", fontsize=9)
ax.set_xlim(0.5, 1.0)
ax.set_xlabel("test AUC")
ax.invert_yaxis()
plt.show()

**Python notu.** `sum(HF_GROUPS.values(), [])` üç sütun listesini, erken füzyonun satırı olan 14 sütunluk tek bir listede toplar. `np.column_stack` üç logit dizisini her hasta için bir satır içeren tek bir dizinin üç sütunu olarak yan yana koyar. Logitlerin ortalaması için model kurmak gerekmez. İstifleme yalnızca doğrulama hastalarında kurulur; çünkü üç model eğitim hastalarında kurulmuştur: Kendi eğitim verilerindeki olasılıkları gereğinden iyidir ve orada kurulan bir istifleme modeli onlara gereğinden fazla güvenirdi. `LogisticRegression(max_iter=5000)` varsayılan düzenlileştirmesini korur.

**Gözlemle ve düşün.** Erken füzyon 14, laboratuvar ise 3 değer kullanır. Daha fazla değer neden daha iyi bir test AUC değeri vermeyebilir ve istiflemedeki EKG ağırlığı neye işaret eder?

**Alıştırma B.** Yalnızca ekokardiyogram ile laboratuvarın, iki logitlerinin ortalamasıyla yapılan geç füzyonunun test AUC değeri kaçtır? Üç ondalık basamakla saklayın.

In [ ]:
veri = alistirma_tablosu("B")   # bu alıştırmanın promptundaki tablo (etkileşimli labın F bölümü)
cevap = None   # cevabınız
teach.check("Alıştırma B", cevap)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
z = (logit(probs_hf["eko"]) + logit(probs_hf["laboratuvar"])) / 2
cevap = round(float(roc_auc_score(yr[rte], z[rte])), 3)
teach.check("Alıştırma B", cevap)

## 3. Farktan ne kadar emin olabiliriz?

*Ders: [Aralıklar: Bir farktan ne kadar emin olunabilir?](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-08/ders.html#intervals-how-sure-is-a-difference)*

Test kümesinde yalnızca birkaç düzine olay vardır. Bootstrap, test hastalarını yerine koyarak tekrar tekrar çeker ve her örnekte iki model aynı hastalarda değerlendirilir; böylece iki AUC değeri arasındaki fark bir aralık kazanır.

**Formüllerden koda.** Bir örnekteki fark: `roc_auc_score(y[i], p1[i]) - roc_auc_score(y[i], p2[i])`. 2.000 fark: `paired_bootstrap(y, p1, p2)`. Yüzde 95 aralık: `np.percentile(d, [2.5, 97.5])`.

**Buradan mı başlıyorsunuz?** Aşağıdaki telafi hücresini bir kez çalıştırın. Hücre, 3. bölümün ihtiyaç duyduğu önceki bölümlerin kodunu, çıktılarını göstermeden çalıştırır.

In [ ]:
#@title Telafi: Jupyter Notebook'a buradan başlıyorsanız bu hücreyi bir kez çalıştırın
def _catch_up(upto, name="NB08_cok_modlu_yapay_zeka_ve_veri_fuzyonu.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-08/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Telafi tamamlandı: Önceki bölümlerin kodu çalıştırıldı.")


_catch_up(3)

In [ ]:
diffs_hf = {}
for name in list(HF_GROUPS) + ["late", "stacked"]:
    d = paired_bootstrap(yr[rte], probs_hf["early"][rte], probs_hf[name][rte])
    diffs_hf[name] = d
    lo, hi = np.percentile(d, [2.5, 97.5])
    print(f"erken füzyon eksi {name:14} {d.mean():+.3f}  ({lo:+.3f} ile {hi:+.3f})  0'ın üstündeki pay {(d > 0).mean():.3f}")
fig, ax = plt.subplots(figsize=(6.4, 2.8))
for i, (name, d) in enumerate(diffs_hf.items()):
    lo, hi = np.percentile(d, [2.5, 97.5])
    ax.plot([lo, hi], [i, i], color="#1F5F8B", lw=2)
    ax.plot(d.mean(), i, "o", color="#1F5F8B")
ax.axvline(0, color="grey", ls="--")
ax.set_yticks(range(len(diffs_hf)))
ax.set_yticklabels(list(diffs_hf))
ax.set_xlabel("erken füzyonun AUC değeri eksi modelin AUC değeri")
ax.invert_yaxis()
plt.show()

**Python notu.** `paired_bootstrap`, `rng.integers` ile test hastası sayısı kadar hastayı yerine koyarak çeker; olay içermeyen ya da olaysız hasta içermeyen bir örnek AUC veremez ve `while` döngüsü tarafından atlanır. İki model aynı çekilmiş hastalar üzerinde puanlanır; bu da ortak şanslarını farkın dışında tutar. `np.random.default_rng` için kullanılan 2026 seed değeri her çalıştırmanın aynı örnekleri çekmesini sağlar. Grafikteki her modelin çizgisi farkın yüzde 95 aralığıdır; bu çizginin 0'daki kesikli çizgiyi kestiği yerde veriler karar vermez.

**Gözlemle ve düşün.** 34 test olayıyla AUC farkının, aralığı artık 0'ı kesmeyecek kadar büyük olması için ne kadar büyük olması gerekirdi? AUC değeri daha yüksek olduğu için füzyonun daha iyi olduğunu bildiren bir meslektaşınıza ne söylerdiniz?

**Alıştırma C.** Test hastalarından çekilen 2.000 bootstrap örneğinin ne kadarında tek başına ekokardiyogram, tek başına EKG'den daha yüksek bir AUC değerine ulaşır? Üç ondalık basamakla saklayın.

In [ ]:
veri = alistirma_tablosu("C")   # bu alıştırmanın promptundaki tablo (etkileşimli labın F bölümü)
cevap = None   # cevabınız
teach.check("Alıştırma C", cevap)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
cevap = round(float((paired_bootstrap(yr[rte], probs_hf["eko"][rte], probs_hf["ECG"][rte]) > 0).mean()), 3)
teach.check("Alıştırma C", cevap)

**1. uygulama durağının Python yolu tamamlandı.** [Ders sayfasına dönün](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-08/ders.html#last-two-hours-cardiovascular-risk-scoring-system) ve okumaya «Son 2 saat: Kardiyovasküler risk skorlama sistemi» bölümünden devam edin.

## 4. Göğüs ağrısı ünitesi ve HEART skoru

*Ders: [Şüpheli AKS erken uyarı modeli](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-08/ders.html#a-risk-model-for-triage-heart-and-the-fused-models)*

Dersin göğüs ağrısı ünitesi olan A hastanesinde göğüs ağrısıyla yapılmış 4.000 başvuru vardır; başvurular numaralarına göre 2.400 eğitim, 800 doğrulama ve 800 test başvurusu olarak ayrılır. B hastanesinde 1.000 başvuru vardır. Sonuç değişkeni 30 gün içindeki majör olumsuz kardiyak olaydır (MACE). Her başvurunun HEART skoru modeller için çıtadır.

**Formüllerden koda.** HEART skorunun her biri 0-2 puanlık beş öğesi: `heart_h`, `heart_e`, `heart_a`, `heart_r` ve `heart_t`; toplamları: `heart`. 0-3, 4-6 ve 7-10 grupları: `pd.cut(..., [-1, 3, 6, 10])`.

**Buradan mı başlıyorsunuz?** Aşağıdaki telafi hücresini bir kez çalıştırın. Hücre, 4. bölümün ihtiyaç duyduğu önceki bölümlerin kodunu, çıktılarını göstermeden çalıştırır.

In [ ]:
#@title Telafi: Jupyter Notebook'a buradan başlıyorsanız bu hücreyi bir kez çalıştırın
def _catch_up(upto, name="NB08_cok_modlu_yapay_zeka_ve_veri_fuzyonu.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-08/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Telafi tamamlandı: Önceki bölümlerin kodu çalıştırıldı.")


_catch_up(4)

In [ ]:
chest = pd.concat([triage.presentations("A"), triage.presentations("B")], ignore_index=True)
A = chest[chest["hospital"] == "A"].reset_index(drop=True)
B = chest[chest["hospital"] == "B"].reset_index(drop=True)
tr, va, te = [(A["set"] == s).to_numpy() for s in ("training", "validation", "test")]
y, yB = A["mace30"].to_numpy(), B["mace30"].to_numpy()

print(chest.groupby(["hospital", "set"])["mace30"].agg(["size", "sum", "mean"]).round(3))
print(A.loc[A["patient_id"] == "CP-3304", ["age", "sex", "hours", "ecg_reading", "troponin_0h", "troponin_1h", "troponin_url", "lvef", "rwma", "heart", "diagnosis", "mace_type"]].T)
print(A.loc[A["patient_id"] == "CP-3304", "note_tr"].iloc[0])

**Python notu.** `triage.presentations("A")` ve `("B")` iki hastanenin başvurularını oluşturur; `pd.concat(..., ignore_index=True)` bunları 0'dan numaralanan tek bir tabloda, `chest` tablosunda, alt alta koyar. Ardından iki hastane `A` ve `B` olarak yeniden ayrılır ve her biri 0'dan yeniden numaralanır; böylece tabloların satırları ve onlardan oluşturulan her dizi hizalı kalır. `tr`, `va` ve `te`, A hastanesinin kümelerinin maskeleridir; `y` ve `yB` ise dizi olarak sonuçlardır. `groupby(["hospital", "set"])` her kümedeki başvuruları ve olayları sayar.

In [ ]:
heart = A["heart"].to_numpy()
group = pd.cut(A["heart"], [-1, 3, 6, 10], labels=["0-3", "4-6", "7-10"])
print(A[te].groupby(group[te], observed=True)["mace30"].agg(["size", "sum", "mean"]).round(3))
auc_heart = roc_auc_score(y[te], heart[te])
print("HEART, test AUC", round(auc_heart, 3), "| yüzde 95 aralığı", np.round(np.percentile(bootstrap_auc(y[te], heart[te].astype(float)), [2.5, 97.5]), 3))
fig, ax = plt.subplots(figsize=(6, 2.8))
share = A[te].groupby(A.loc[te, "heart"])["mace30"].mean()
ax.bar(share.index, share.values, color=["#0E7A78"] * 4 + ["#B97813"] * 3 + ["#9B1B1B"] * 4)
ax.set_xlabel("HEART skoru")
ax.set_ylabel("MACE oranı")
plt.show()

**Python notu.** `pd.cut` her skoru düştüğü aralığa yerleştirir: (−1, 3], (3, 6] ve (6, 10]; etiketleri 0-3, 4-6 ve 7-10'dur. `groupby(group[te], observed=True)` test başvurularını bu etiketlere göre gruplar; `agg(["size", "sum", "mean"])` de her gruptaki başvuru sayısını, olay sayısını ve olayların oranını verir. Tam sayı bir skor `roc_auc_score` fonksiyonuna doğrudan girebilir: AUC yalnızca hastaların sırasını kullanır. Aralığı 1. bölümdeki `bootstrap_auc` verir.

**Gözlemle ve düşün.** MACE oranı HEART skoruyla ne kadar dik yükseliyor? Taburculuk çizgisini nereye koyardınız ve bu çizginin altında kaç olayı kabul ederdiniz?

**Alıştırma D.** HEART skoru 0-3 olan test başvurularının ne kadarında MACE görülmüştür? Oranı üç ondalık basamakla saklayın.

In [ ]:
veri = alistirma_tablosu("D")   # bu alıştırmanın promptundaki tablo (etkileşimli labın F bölümü)
cevap = None   # cevabınız
teach.check("Alıştırma D", cevap)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
cevap = round(float(y[te][heart[te] <= 3].mean()), 3)
teach.check("Alıştırma D", cevap)

## 5. Beş modalite, beş model

*Ders: [Bir kalp ekibinin verileri: Sanal kliniğin göğüs ağrısı ünitesi](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-08/ders.html#the-data-of-a-heart-team-the-chest-pain-unit-of-the-virtual-clinic)*

Her modalite bir sayı tablosuna dönüşür: Klinik tablo (12 değer), testinin üst referans sınırının katları olarak troponin (4 değer), eksik bir çalışmayı işaretleyen sütunlarla ekokardiyogram (3 değer) ve kelimelerin ve kelime çiftlerinin 20 sayıya indirilmiş TF-IDF değerleri olarak Türkçe triyaj notu. Her biri üzerinde bir lojistik regresyon kurulur. EKG 6. bölümde ele alınır.

**Formüllerden koda.** Her modalitenin $x_m$ değerleri: `clinical(d)`, `laboratory(d)`, `echo(d)` ve `note(d)`. İki hastanedeki $p_m$ olasılıkları: `P[name]` ve `PB[name]`.

**Buradan mı başlıyorsunuz?** Aşağıdaki telafi hücresini bir kez çalıştırın. Hücre, 5. bölümün ihtiyaç duyduğu önceki bölümlerin kodunu, çıktılarını göstermeden çalıştırır.

In [ ]:
#@title Telafi: Jupyter Notebook'a buradan başlıyorsanız bu hücreyi bir kez çalıştırın
def _catch_up(upto, name="NB08_cok_modlu_yapay_zeka_ve_veri_fuzyonu.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-08/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Telafi tamamlandı: Önceki bölümlerin kodu çalıştırıldı.")


_catch_up(5)

In [ ]:
CLINICAL = ["age", "male", "smoker", "diabetes", "hypertension", "hyperlipidaemia", "obesity", "family_history", "known_cad",
            "heart_rate", "sbp", "egfr"]


def clinical(d):
    """Klinik tablo: Yaş, cinsiyet, risk faktörleri, bilinen koroner arter hastalığı, kalp hızı, sistolik basınç ve eGFR."""
    return d.assign(male=(d["sex"] == "M").astype(int))[CLINICAL].astype(float)


def laboratory(d):
    """Kendi testinin üst referans sınırının katları olarak troponin: Gelişteki ve bir saat sonraki değerin logaritması, bir saatteki değişim ve ikinci değerin olup olmadığı."""
    url = d["troponin_url"]
    t0, t1 = d["troponin_0h"] / url, d["troponin_1h"] / url
    return pd.DataFrame({"log_trop_0h": np.log(t0), "log_trop_1h": np.log(t1.fillna(t0)),
                         "delta_1h": (t1 - t0).fillna(0.0), "has_1h": t1.notna().astype(float)})


def echo(d):
    """Yatak başı ekokardiyogram: Yapılıp yapılmadığı, ejeksiyon fraksiyonu (yapılmadıysa 60) ve bölgesel duvar hareket bozukluğu (yapılmadıysa 0)."""
    return pd.DataFrame({"echo_done": d["echo_done"].astype(float), "lvef": d["lvef"].fillna(60.0), "rwma": d["rwma"].fillna(0.0)})

tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=3, sublinear_tf=True).fit(A.loc[tr, "note_tr"])
svd = TruncatedSVD(n_components=20, random_state=RANDOM_STATE).fit(tfidf.transform(A.loc[tr, "note_tr"]))


def note(d):
    """Triyaj notu 20 sayı olarak: Kelimelerin ve kelime çiftlerinin TF-IDF değerleri, eğitim notlarında öğrenilen bir tekil değer ayrışımıyla küçültülür."""
    return pd.DataFrame(svd.transform(tfidf.transform(d["note_tr"])), columns=[f"note_{i + 1}" for i in range(20)])

tables = {"klinik": (clinical(A), clinical(B)), "troponin": (laboratory(A), laboratory(B)),
          "ekokardiyogram": (echo(A), echo(B)), "not": (note(A), note(B))}
P, PB = {}, {}
for name, (X, XB) in tables.items():
    model = fit_logistic(X[tr], y[tr], Xv=X[va], yv=y[va])
    P[name], PB[name] = model.predict_proba(X)[:, 1], model.predict_proba(XB)[:, 1]
    print(f"{name:16} {X.shape[1]:3} değer   test AUC {roc_auc_score(y[te], P[name][te]):.3f}   B hastanesi {roc_auc_score(yB, PB[name]):.3f}")

**Python notu.** Dört fonksiyonun her biri bir başvuru tablosu alır ve aynı sırada, her başvuru için bir satır olmak üzere bir modalitenin sayılarını döndürür: A hastanesi için `clinical(A)`, B hastanesi için `clinical(B)`. `laboratory` her troponini kendi testinin üst referans sınırına böler; böylece A'nın (troponin T) ve B'nin (troponin I) değerleri aynı anlamı taşır. `note` içindeki TF-IDF ve tekil değer ayrışımı yalnızca eğitim başvurularının notlarında öğrenilir, sonra bütün notlara uygulanır. `tables` sözlüğü her modalitenin tablo çiftini tutar ve döngü her biri için bir model kurar.

In [ ]:
words = tfidf.get_feature_names_out()
weights = LogisticRegression(max_iter=5000).fit(tfidf.transform(A.loc[tr, "note_tr"]), y[tr]).coef_[0]
order = np.argsort(weights)
print("riski düşüren kelimeler:", list(words[order[:8]]))
print("riski artıran kelimeler:", list(words[order[-8:]]))

**Python notu.** `get_feature_names_out` TF-IDF'nin tuttuğu kelimeleri ve kelime çiftlerini verir. 20 sayıya indirmeden hepsi üzerinde kurulan sade bir lojistik regresyon, hangi terimlerin riski yukarı ya da aşağı ittiğini gösterir: `np.argsort` ağırlıkları sıralar; ilkler en negatif, sonuncular en pozitif ağırlıklardır. Notlar Türkçe yazıldığı için terimler Jupyter Notebook'un iki sürümünde de Türkçedir. Olumsuzluklar önemlidir: "yoktur" ve "değildir" bir özelliği tersine çevirir.

**Gözlemle ve düşün.** Not tek başına klinik tabloyla aşağı yukarı aynı AUC değerine ulaşıyor. En çok ağırlığı hangi kelimeler taşıyor ve notları kısa olan B hastanesinde not neden bütün modalitelerden daha çok kaybediyor?

**Alıştırma E.** Eğitim notlarının TF-IDF'si kaç kelime ve kelime çifti tutar, `tfidf.vocabulary_`? Sayıyı saklayın.

In [ ]:
cevap = None   # cevabınız
teach.check("Alıştırma E", cevap)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
cevap = len(tfidf.vocabulary_)
teach.check("Alıştırma E", cevap)

## 6. Tek boyutlu evrişimli bir ağla EKG

*Ders: [Bir kalp ekibinin verileri: Sanal kliniğin göğüs ağrısı ünitesi](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-08/ders.html#the-data-of-a-heart-team-the-chest-pain-unit-of-the-virtual-clinic)*

Her başvurunun 100 Hz'de 10 saniyelik 12 derivasyonlu EKG'si 5. haftadakine benzer tek boyutlu evrişimli bir ağa girer: Zaman boyunca evrişimlerden oluşan dört blok, her kanalın zaman boyunca ortalaması ve tek bir çıktı. Çıktıdan önceki 64 sayı traseyi tarif eder ve 7. bölümdeki füzyon modellerinin EKG modalitesi olur.

**Formüllerden koda.** Traseler: `ecg_A` ve `ecg_B`; başvuru × 12 derivasyon × 1.000 örnek boyutlu diziler. $p_{\mathrm{EKG}}$ olasılığı: `ecg_predict(ecgnet, ...)`; 64 sayı: `ecg_embed(ecgnet, ...)`.

**Buradan mı başlıyorsunuz?** Aşağıdaki telafi hücresini bir kez çalıştırın. Hücre, 6. bölümün ihtiyaç duyduğu önceki bölümlerin kodunu, çıktılarını göstermeden çalıştırır.

In [ ]:
#@title Telafi: Jupyter Notebook'a buradan başlıyorsanız bu hücreyi bir kez çalıştırın
def _catch_up(upto, name="NB08_cok_modlu_yapay_zeka_ve_veri_fuzyonu.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-08/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Telafi tamamlandı: Önceki bölümlerin kodu çalıştırıldı.")


_catch_up(6)

In [ ]:
ecg_A = triage.ecg_array(A)          # 12 derivasyonlu 4.000 EKG, 100 Hz'de 10 saniye: 4000 x 12 x 1000
ecg_B = triage.ecg_array(B)

print(ecg_A.shape, ecg_B.shape)
i = int(np.where(A["patient_id"] == "CP-3304")[0][0])
t = np.arange(1000) / 100
fig, ax = plt.subplots(3, 1, figsize=(9, 4.2), sharex=True)
for a, lead in zip(ax, [1, 6, 10]):
    a.plot(t, ecg_A[i, lead], color="#231A1A", lw=0.8)
    a.set_ylabel(vclinic.LEADS[lead])
ax[-1].set_xlabel("saniye")
plt.show()

**Python notu.** `triage.ecg_array` her EKG'yi 500 Hz'de çizer ve her beşinci örneği, yani saniyede 100 örneği tutar; bu, ST segmenti ve T dalgası için yeterlidir ve öğrenmek için beş kat daha küçüktür. Diziler `A` ve `B` tablolarının satırlarını izler; böylece `ecg_A[i]`, `A` tablosunun `i`. satırındaki başvurunun EKG'sidir: `np.where(A["patient_id"] == "CP-3304")[0][0]` bu satırı bulur. Python 0'dan saydığı için on iki derivasyon arasında 1, 6 ve 10 numaralı olanlar DII, V1 ve V5'tir.

In [ ]:
class ECGNet(nn.Module):
    """12 derivasyonlu bir EKG için tek boyutlu evrişimli ağ: Dört blok, her kanalın zaman boyunca ortalaması ve tek bir çıktı."""
    def __init__(self):
        super().__init__()
        block = lambda a, b: nn.Sequential(nn.Conv1d(a, b, 7, padding=3), nn.BatchNorm1d(b), nn.ReLU(), nn.MaxPool1d(2))
        self.features = nn.Sequential(block(12, 32), block(32, 64), block(64, 64), block(64, 64))
        self.head = nn.Linear(64, 1)

    def embed(self, x):
        return self.features(x).mean(dim=2)

    def forward(self, x):
        return self.head(self.embed(x)).squeeze(1)


def train_ecg(model, X, y, Xv, yv, epochs=12, seed=2026, verbose=True):
    """64 EKG'lik gruplarla, öğrenme hızı 0,001 olan Adam; her EKG zamanda en fazla yarım saniye kaydırılır. Doğrulama AUC değeri en iyi olan epoch'un ağırlıkları saklanır."""
    torch.manual_seed(seed)
    X, y = torch.as_tensor(X), torch.as_tensor(y, dtype=torch.float32)
    optimiser = torch.optim.Adam(model.parameters(), lr=0.001)
    loss_fn = nn.BCEWithLogitsLoss()
    best, state = -1.0, None
    for epoch in range(epochs):
        model.train()
        order = torch.randperm(len(X))
        for i in range(0, len(X), 64):
            b = order[i:i + 64]
            xb = torch.roll(X[b], int(torch.randint(-50, 51, (1,))), dims=2)
            loss = loss_fn(model(xb), y[b])
            optimiser.zero_grad()
            loss.backward()
            optimiser.step()
        a = roc_auc_score(yv, ecg_predict(model, Xv))
        if a > best:
            best, state = a, copy.deepcopy(model.state_dict())
        if verbose:
            print(f"epoch {epoch + 1}: doğrulama AUC {a:.3f}")
    model.load_state_dict(state)
    return best


@torch.no_grad()
def ecg_predict(model, X, batch=256):
    model.eval()
    return torch.cat([torch.sigmoid(model(torch.as_tensor(X[i:i + batch]))) for i in range(0, len(X), batch)]).numpy()


@torch.no_grad()
def ecg_embed(model, X, batch=256):
    model.eval()
    return torch.cat([model.embed(torch.as_tensor(X[i:i + batch])) for i in range(0, len(X), batch)]).numpy()

torch.manual_seed(RANDOM_STATE)
ecgnet = ECGNet()
print("ağırlıklar:", sum(p.numel() for p in ecgnet.parameters()))
train_ecg(ecgnet, ecg_A[tr], y[tr], ecg_A[va], y[va])
P["EKG ağı"], PB["EKG ağı"] = ecg_predict(ecgnet, ecg_A), ecg_predict(ecgnet, ecg_B)
print("EKG ağı, test AUC", round(roc_auc_score(y[te], P["EKG ağı"][te]), 3), "| B hastanesi", round(roc_auc_score(yB, PB["EKG ağı"]), 3))
for reading in ["normal", "T-wave inversion", "ST depression", "ST elevation", "LBBB"]:
    m = (A["ecg_reading"] == reading).to_numpy() & te
    print(f"{reading:18} ortalama olasılık {P['EKG ağı'][m].mean():.3f}")

**Python notu.** `nn.Conv1d(a, b, 7, padding=3)` 7 örnekten, yani 70 ms'den oluşan bir pencereyi `a` kanal üzerinde zaman boyunca kaydırır ve `b` kanal verir; ilk blok 12 derivasyonu okur. `nn.MaxPool1d(2)` her bloktan sonra zamanı yarıya indirir; böylece dört blok 1.000 örneği 62 örneğe dönüştürür. `embed` her kanalın zaman boyunca ortalamasını alır ve 64 sayı verir; `head` de bunları tek bir logite dönüştürür. `train_ecg` her traseyi en fazla yarım saniye kaydırır (`torch.roll`); böylece ağ bir atımın nereye düştüğünü öğrenemez. Doğrulama AUC değeri en iyi olan epoch'un ağırlıklarını tutar. Okumaya göre ortalama olasılık, ağın ne öğrendiğini gösterir: ST segmenti ve T dalgası riski değiştirir.

**Gözlemle ve düşün.** Ağ EKG'nin okumasını hiç görmemiş, yalnızca traseyi görmüştür. Ortalama olasılığı okumaları bir kardiyoloğun beklediği sırayla izliyor mu? Bir traseyi bir kardiyolog kadar iyi okuyabilmesi için neye ihtiyacı olurdu?

**Alıştırma F.** EKG ağı `ecgnet` kaç eğitilebilir ağırlık taşır?

In [ ]:
cevap = None   # cevabınız
teach.check("Alıştırma F", cevap)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
cevap = sum(p.numel() for p in ecgnet.parameters())
teach.check("Alıştırma F", cevap)

## 7. Erken, geç, hibrit ve dikkat tabanlı füzyon

*Ders: [Erken füzyon vs geç füzyon vs hibrit füzyon](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-08/ders.html#early-late-and-hybrid-fusion)*

Bu bölümde aynı beş modalite üzerinde beş tür füzyon kurulur: 103 sayının tamamı üzerinde tek bir lojistik regresyon olan erken füzyon; beş logitin ortalamasıyla ve istiflemeyle geç füzyon; her modalite için bir kodlayıcısı ve ortak bir karar katmanı olan hibrit bir ağ; ve her modalite için bir token'ı, bir sınıf token'ı, eksik ekokardiyogram için maskeleri ve modalite dropout'u olan bir dikkat ağı.

**Formüllerden koda.** $z = \beta_0 + \beta^{\top}[\,x_1; \ldots; x_5\,]$: `early`. $\frac{1}{5}\sum_m z_m$: `L.mean(axis=1)`. $b + \sum_m w_m z_m$: `stack`. $g([f_1(x_1); \ldots])$: `HybridFusion`. Maske $k_m$: `keep`; modalite dropout'u: `drop=0.15`.

**Buradan mı başlıyorsunuz?** Aşağıdaki telafi hücresini bir kez çalıştırın. Hücre, 7. bölümün ihtiyaç duyduğu önceki bölümlerin kodunu, çıktılarını göstermeden çalıştırır.

In [ ]:
#@title Telafi: Jupyter Notebook'a buradan başlıyorsanız bu hücreyi bir kez çalıştırın
def _catch_up(upto, name="NB08_cok_modlu_yapay_zeka_ve_veri_fuzyonu.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-08/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Telafi tamamlandı: Önceki bölümlerin kodu çalıştırıldı.")


_catch_up(7)

In [ ]:
MODALITIES = ["klinik", "troponin", "ekokardiyogram", "EKG ağı", "not"]


class Attention(nn.Module):
    """Çok başlı öz dikkat; bir maske eksik modalitelerin token'larını çıkarır."""
    def __init__(self, dim, heads):
        super().__init__()
        self.heads = heads
        self.qkv = nn.Linear(dim, 3 * dim)
        self.out = nn.Linear(dim, dim)
        self.weights = None

    def forward(self, x, keep):
        n, t, d = x.shape
        q, k, v = self.qkv(x).reshape(n, t, 3, self.heads, d // self.heads).permute(2, 0, 3, 1, 4)
        s = q @ k.transpose(-2, -1) / (d // self.heads) ** 0.5
        s = s.masked_fill(~keep[:, None, None, :], float("-inf"))
        a = torch.softmax(s, dim=-1)
        self.weights = a.detach()
        return self.out((a @ v).transpose(1, 2).reshape(n, t, d))


class Block(nn.Module):
    """7. haftanın Transformer bloğu; maske dikkate aktarılır."""
    def __init__(self, dim, heads, hidden):
        super().__init__()
        self.norm1, self.attn = nn.LayerNorm(dim), Attention(dim, heads)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(nn.Linear(dim, hidden), nn.GELU(), nn.Linear(hidden, dim))

    def forward(self, x, keep):
        x = x + self.attn(self.norm1(x), keep)
        return x + self.mlp(self.norm2(x))


class HybridFusion(nn.Module):
    """Ara füzyon: Her modalite için küçük bir kodlayıcı, çıktıları yan yana ve ortak bir karar katmanı."""
    def __init__(self, dims, dim=32):
        super().__init__()
        self.enc = nn.ModuleList([nn.Sequential(nn.Linear(d, dim), nn.ReLU()) for d in dims])
        self.head = nn.Sequential(nn.Linear(dim * len(dims), dim), nn.ReLU(), nn.Dropout(0.2), nn.Linear(dim, 1))

    def forward(self, xs, keep):
        h = [e(x) * k[:, None] for e, x, k in zip(self.enc, xs, keep.T)]
        return self.head(torch.cat(h, dim=1)).squeeze(1)


class AttentionFusion(nn.Module):
    """Dikkat tabanlı füzyon: Her modalite için bir token ve bir sınıf token'ı iki Transformer bloğundan geçer; eksik modaliteler maskelenir."""
    def __init__(self, dims, dim=32, depth=2, heads=4):
        super().__init__()
        self.enc = nn.ModuleList([nn.Linear(d, dim) for d in dims])
        self.kind = nn.Parameter(torch.randn(1, len(dims), dim) * 0.02)
        self.cls = nn.Parameter(torch.zeros(1, 1, dim))
        self.blocks = nn.ModuleList([Block(dim, heads, 2 * dim) for _ in range(depth)])
        self.norm, self.head = nn.LayerNorm(dim), nn.Linear(dim, 1)

    def forward(self, xs, keep):
        t = torch.stack([e(x) for e, x in zip(self.enc, xs)], dim=1) + self.kind
        t = torch.cat([self.cls.expand(len(t), -1, -1), t], dim=1)
        keep = torch.cat([torch.ones(len(t), 1, dtype=torch.bool), keep], dim=1)
        for block in self.blocks:
            t = block(t, keep)
        return self.head(self.norm(t[:, 0])).squeeze(1)


def train_fusion(model, xs, keep, y, xs_v, keep_v, yv, epochs=40, seed=2026, drop=0.0, verbose=False):
    """64 hastalık gruplarla Adam; drop > 0 olduğunda eğitim sırasında var olan her modalite rastgele gizlenir (modalite dropout'u). Doğrulama AUC değeri en iyi olan epoch'un ağırlıkları saklanır."""
    torch.manual_seed(seed)
    xs = [torch.as_tensor(x, dtype=torch.float32) for x in xs]
    keep, y = torch.as_tensor(keep, dtype=torch.bool), torch.as_tensor(y, dtype=torch.float32)
    optimiser = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
    loss_fn = nn.BCEWithLogitsLoss()
    best, state = -1.0, None
    for epoch in range(epochs):
        model.train()
        order = torch.randperm(len(y))
        for i in range(0, len(y), 64):
            b = order[i:i + 64]
            kb = keep[b]
            if drop > 0:
                kb = kb & (torch.rand(kb.shape) >= drop)
                kb[kb.sum(dim=1) == 0, 0] = True
            loss = loss_fn(model([x[b] for x in xs], kb), y[b])
            optimiser.zero_grad()
            loss.backward()
            optimiser.step()
        a = roc_auc_score(yv, fusion_predict(model, xs_v, keep_v))
        if a > best:
            best, state = a, copy.deepcopy(model.state_dict())
        if verbose:
            print(f"epoch {epoch + 1}: doğrulama AUC {a:.3f}")
    model.load_state_dict(state)
    return best


@torch.no_grad()
def fusion_predict(model, xs, keep):
    model.eval()
    xs = [torch.as_tensor(x, dtype=torch.float32) for x in xs]
    return torch.sigmoid(model(xs, torch.as_tensor(keep, dtype=torch.bool))).numpy()

emb = lambda X: pd.DataFrame(ecg_embed(ecgnet, X), columns=[f"ecg_{i + 1}" for i in range(64)])
parts_A = [clinical(A), laboratory(A), echo(A), emb(ecg_A), note(A)]
parts_B = [clinical(B), laboratory(B), echo(B), emb(ecg_B), note(B)]
XE, XEB = pd.concat(parts_A, axis=1), pd.concat(parts_B, axis=1)
early = fit_logistic(XE[tr], y[tr], Xv=XE[va], yv=y[va])
P["early"], PB["early"] = early.predict_proba(XE)[:, 1], early.predict_proba(XEB)[:, 1]
L, LB = np.column_stack([logit(P[m]) for m in MODALITIES]), np.column_stack([logit(PB[m]) for m in MODALITIES])
P["late"], PB["late"] = 1 / (1 + np.exp(-L.mean(axis=1))), 1 / (1 + np.exp(-LB.mean(axis=1)))
stack = LogisticRegression(max_iter=5000).fit(L[va], y[va])
P["stacked"], PB["stacked"] = stack.predict_proba(L)[:, 1], stack.predict_proba(LB)[:, 1]
print(XE.shape, "| istifleme ağırlıkları:", dict(zip(MODALITIES, np.round(stack.coef_[0], 3))), "| sabit terim", round(stack.intercept_[0], 3))

**Python notu.** `pd.concat(parts_A, axis=1)` beş tabloyu erken füzyonun satırında yan yana koyar: Başvuru başına 12 + 4 + 3 + 64 + 20 = 103 sayı. EKG erken füzyona ağının 64 sayısıyla, geç füzyona ise ağın olasılığıyla girer. `L`, her başvurunun beş logitini `MODALITIES` sırasıyla tutar; aynı sıra `stack` modelinin beş ağırlığını da verir.

In [ ]:
scalers = [StandardScaler().fit(X[tr]) for X in parts_A]
xs = [s.transform(X).astype(np.float32) for s, X in zip(scalers, parts_A)]
xsB = [s.transform(X).astype(np.float32) for s, X in zip(scalers, parts_B)]
keep = np.ones((len(A), 5), dtype=bool)
keep[:, 2] = A["echo_done"].to_numpy() == 1
keepB = np.ones((len(B), 5), dtype=bool)
keepB[:, 2] = B["echo_done"].to_numpy() == 1
dims = [x.shape[1] for x in xs]
nets = {}
for name, Net, drop in (("hybrid", HybridFusion, 0.0), ("attention", AttentionFusion, 0.15)):
    torch.manual_seed(RANDOM_STATE)
    nets[name] = Net(dims)
    train_fusion(nets[name], [x[tr] for x in xs], keep[tr], y[tr], [x[va] for x in xs], keep[va], y[va], drop=drop)
    P[name], PB[name] = fusion_predict(nets[name], xs, keep), fusion_predict(nets[name], xsB, keepB)
    print(name, "| ağırlık", sum(p.numel() for p in nets[name].parameters()), "| test AUC", round(roc_auc_score(y[te], P[name][te]), 3))

**Python notu.** Her modalite kendi eğitim değerleriyle standartlaştırılır: `StandardScaler().fit(X[tr])`. `keep` maskedir: Başvuru × 5 modalite boyutlu bir tablodur ve modalite varsa `True` değerini taşır; burada yalnızca ekokardiyogram eksik olabilir. Hibrit ağ eksik bir kodlayıcının çıktısını 0 ile çarpar; dikkat ağı ise eksik bir modalitenin token'ına eksi sonsuz puan verir. `drop=0.15` ile `train_fusion` ayrıca bir eğitim batch'indeki var olan her modaliteyi yüzde 15 olasılıkla gizler, ama hiçbir zaman hepsini birden gizlemez. İki ağ da doğrulamadaki en iyi epoch'un ağırlıklarını tutar.

In [ ]:
rows = []
for name in MODALITIES + ["early", "late", "stacked", "hybrid", "attention"]:
    lo, hi = np.percentile(bootstrap_auc(y[te], P[name][te]), [2.5, 97.5])
    rows.append([name, round(roc_auc_score(y[te], P[name][te]), 3), round(lo, 3), round(hi, 3)])
lo, hi = np.percentile(bootstrap_auc(y[te], heart[te].astype(float)), [2.5, 97.5])
rows.append(["HEART", round(auc_heart, 3), round(lo, 3), round(hi, 3)])
print(pd.DataFrame(rows, columns=["model", "test AUC", "2.5%", "97.5%"]).to_string(index=False))
i = int(np.where(A["patient_id"] == "CP-3304")[0][0])
print("CP-3304:", {name: round(float(P[name][i]), 3) for name in P})
d = paired_bootstrap(y[te], P["early"][te], heart[te].astype(float))
print("erken füzyon eksi HEART", round(d.mean(), 3), np.round(np.percentile(d, [2.5, 97.5]), 3))

**Python notu.** 1. bölümdeki `bootstrap_auc` her modelin 2.000 AUC değerini, `np.percentile` de aralığını verir; `paired_bootstrap` ise erken füzyonu aynı örneklerde HEART ile karşılaştırır. `{name: ... for name in P}` sözlük üreteci, vakanın, yani `A` tablosunun `i`. satırının her model için olasılığını toplar.

**Gözlemle ve düşün.** Beş füzyon modeli neredeyse aynı AUC değerine ulaşıyor. Bir göğüs ağrısı ünitesi için hangisini seçerdiniz ve seçmeden önce AUC dışında neye bakardınız? CP-3304'ün beş olasılığını karşılaştırın.

**Alıştırma G.** Dikkat tabanlı füzyon ağı `nets["attention"]` kaç eğitilebilir ağırlık taşır?

In [ ]:
cevap = None   # cevabınız
teach.check("Alıştırma G", cevap)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
cevap = sum(p.numel() for p in nets["attention"].parameters())
teach.check("Alıştırma G", cevap)

## 8. Test anında eksik modaliteler

*Ders: [Eksik veri yönetimi: Klinik gerçeklik](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-08/ders.html#missing-modalities)*

Laboratuvar sistemi çalışmadığında ya da triyaj notu yazılmadığında olduğu gibi, bütün bir modalite her test hastasında eksik olursa ne olur? Erken füzyon modaliteyi eğitim ortalamasıyla doldurur; hibrit ağ kodlayıcısının çıktısını 0 ile çarpar; modalite dropout'uyla eğitilmiş dikkat ağı ise token'ını maskeler.

**Formüllerden koda.** Gizlenen modalite için maske $k_m = 0$: `kk[:, j] = False`. Erken füzyondaki eğitim ortalaması: `XE.loc[tr, cols].mean()`.

**Buradan mı başlıyorsunuz?** Aşağıdaki telafi hücresini bir kez çalıştırın. Hücre, 8. bölümün ihtiyaç duyduğu önceki bölümlerin kodunu, çıktılarını göstermeden çalıştırır.

In [ ]:
#@title Telafi: Jupyter Notebook'a buradan başlıyorsanız bu hücreyi bir kez çalıştırın
def _catch_up(upto, name="NB08_cok_modlu_yapay_zeka_ve_veri_fuzyonu.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-08/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Telafi tamamlandı: Önceki bölümlerin kodu çalıştırıldı.")


_catch_up(8)

In [ ]:
rows = []
for j, name in enumerate(MODALITIES):
    kk = keep.copy()
    kk[:, j] = False
    XE_ = XE.copy()
    cols = parts_A[j].columns
    XE_[cols] = XE.loc[tr, cols].mean().to_numpy()
    rows.append([name, round(roc_auc_score(y[te], early.predict_proba(XE_[te])[:, 1]), 3),
                 round(roc_auc_score(y[te], fusion_predict(nets["hybrid"], [x[te] for x in xs], kk[te])), 3),
                 round(roc_auc_score(y[te], fusion_predict(nets["attention"], [x[te] for x in xs], kk[te])), 3)])
missing = pd.DataFrame(rows, columns=["gizlenen", "erken füzyon", "hibrit", "dikkat"])
print(missing.to_string(index=False))
missing.set_index("gizlenen").plot.bar(figsize=(7, 3), ylim=(0.8, 0.95), rot=0, color=["#1F5F8B", "#0E7A78", "#6E46A0"])
plt.ylabel("test AUC")
plt.show()

**Python notu.** `keep.copy()` özgün maskeye dokunmamak için yeni bir maske oluşturur; `kk[:, j] = False` de `j` modalitesini her hasta için gizler. `XE.copy()` erken füzyonun tablosu için aynısını yapar; gizlenen modalitenin sütunlarına eğitim ortalamalarını atamak, `XE_[cols] = ...`, her hastaya bu modalite için ortalama bir hastanın değerlerini verir. Modellerin kendileri değiştirilmez: Bu, her füzyon türünün yatak başında ilk kez karşılaştığı bir boşlukla ne yaptığının bir sınamasıdır.

**Gözlemle ve düşün.** Füzyon modelleri hangi modaliteyi kaybetmeyi en az göze alabilir ve neden? Troponin eksikken bir olasılık veren bir modele güvenir miydiniz ve ünite bunun yerine ne yapmalıdır?

**Alıştırma H.** Triyaj notu her test hastasından gizlendiğinde erken füzyonun test AUC değeri kaçtır? `missing` tablosundan okuyun ve üç ondalık basamakla saklayın.

In [ ]:
veri = alistirma_tablosu("H")   # bu alıştırmanın promptundaki tablo (etkileşimli labın F bölümü)
cevap = None   # cevabınız
teach.check("Alıştırma H", cevap)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
cevap = float(missing.loc[missing["gizlenen"] == "not", "erken füzyon"].iloc[0])
teach.check("Alıştırma H", cevap)

**2. uygulama durağının Python yolu tamamlandı.** [Ders sayfasına dönün](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-08/ders.html#another-hospital-drift-and-recalibration) ve okumaya «Başka bir hastane: Kayma ve yeniden kalibrasyon» bölümünden devam edin.

## 9. Kalibrasyon ve başka bir hastane

*Ders: [Kalibrasyon: Olasılıklar söylediklerini ifade ediyor mu?](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-08/ders.html#calibration-do-the-probabilities-mean-what-they-say)*

AUC hastaların sırasını, kalibrasyon ise sayıları değerlendirir. Bu bölüm A hastanesinin test kümesinde üç füzyon modelinin kalibrasyon sabitini, eğimini ve Brier skorunu ölçer, güvenilirlik eğrilerini çizer; ardından erken füzyonu, ilk 300 hastayla güncellendiği B hastanesine kadar izler.

**Formüllerden koda.** $\mathrm{logit}\,P(y = 1) = a + b\,z$: `calibration(y, p)` $a$ ve $b$ değerlerini döndürür. Brier: `brier_score_loss(y, p)`. Güncelleme: `first` hastalarında öğrenilen $a$ kayması B'nin her logitine eklenir.

**Buradan mı başlıyorsunuz?** Aşağıdaki telafi hücresini bir kez çalıştırın. Hücre, 9. bölümün ihtiyaç duyduğu önceki bölümlerin kodunu, çıktılarını göstermeden çalıştırır.

In [ ]:
#@title Telafi: Jupyter Notebook'a buradan başlıyorsanız bu hücreyi bir kez çalıştırın
def _catch_up(upto, name="NB08_cok_modlu_yapay_zeka_ve_veri_fuzyonu.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-08/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Telafi tamamlandı: Önceki bölümlerin kodu çalıştırıldı.")


_catch_up(9)

In [ ]:
fig, ax = plt.subplots(figsize=(4.8, 4.4))
for name, color in (("early", "#1F5F8B"), ("stacked", "#0E7A78"), ("late", "#B97813")):
    a, b = calibration(y[te], P[name][te])
    print(f"{name:8} ortalama {P[name][te].mean():.3f}  gözlenen {y[te].mean():.3f}  kalibrasyon sabiti {a:+.2f}  eğim {b:.2f}  Brier {brier_score_loss(y[te], P[name][te]):.3f}")
    pred, obs = reliability(y[te], P[name][te])
    ax.plot(pred, obs, "o-", color=color, label=name)
ax.plot([0, 1], [0, 1], "--", color="grey")
ax.set_xlabel("ortalama tahmin edilen risk")
ax.set_ylabel("gözlenen MACE oranı")
ax.legend()
plt.show()

**Python notu.** `calibration`, eğim 1'e sabitlenmişken kalibrasyon sabitini ($a$) Newton adımlarıyla bulur: Her adım, gözlenen ve tahmin edilen olay sayıları arasındaki farkın $p(1 - p)$ toplamına bölümünü ekler. Eğim, sonucun logit üzerine kurulan düzenlileştirmesiz (`C=1e6`) bir lojistik regresyonunun katsayısıdır. `reliability` hastaları risklerine göre sıralar, `np.array_split` ile 80'er hastalık on gruba böler ve her grubun ortalama tahmin edilen riskini ve gözlenen oranını döndürür.

In [ ]:
for name in ["early", "stacked", "hybrid", "attention"]:
    a, b = calibration(yB, PB[name])
    print(f"{name:10} B hastanesi: AUC {roc_auc_score(yB, PB[name]):.3f}  ortalama {PB[name].mean():.3f}  gözlenen {yB.mean():.3f}  kalibrasyon sabiti {a:+.2f}  eğim {b:.2f}")
first, rest = np.arange(len(B)) < 300, np.arange(len(B)) >= 300
zB = logit(PB["early"])
shift = 0.0
for _ in range(50):
    q = 1 / (1 + np.exp(-(shift + zB[first])))
    shift += (yB[first] - q).sum() / (q * (1 - q)).sum()
pB_new = 1 / (1 + np.exp(-(zB + shift)))
print("ilk 300 hastada öğrenilen kayma:", round(shift, 3))
fig, ax = plt.subplots(figsize=(4.8, 4.4))
for p, label, color in ((PB["early"], "önce", "#9B1B1B"), (pB_new, "sonra", "#1F5F8B")):
    a, b = calibration(yB[rest], p[rest])
    print(f"{label:7} ortalama {p[rest].mean():.3f}  gözlenen {yB[rest].mean():.3f}  kalibrasyon sabiti {a:+.2f}  eğim {b:.2f}  Brier {brier_score_loss(yB[rest], p[rest]):.3f}  AUC {roc_auc_score(yB[rest], p[rest]):.3f}")
    pred, obs = reliability(yB[rest], p[rest])
    ax.plot(pred, obs, "o-", color=color, label=label)
ax.plot([0, 1], [0, 1], "--", color="grey")
ax.set_xlabel("ortalama tahmin edilen risk")
ax.set_ylabel("gözlenen MACE oranı")
ax.set_title("B hastanesi, ilk 300 hastadan sonraki 700 hasta", fontsize=10)
ax.legend()
plt.show()

**Python notu.** B'nin ilk 300 başvurusu (`first`) kullanımın ilk aylarını, öteki 700 başvuru (`rest`) ise güncellemeden sonrasını temsil eder. Döngü `calibration` içindeki Newton aramasının aynısıdır, ancak yalnızca ilk 300 başvuruda çalışır: Bu başvuruların ortalama tahmin edilen riskini olay oranlarına eşitleyen kaymayı bulur. Kaymayı B'nin her logitine eklemek (`zB + shift`) her hastayı aynı adımla yukarı taşır; bu yüzden hastaların sırası ve AUC değişmez.

**Gözlemle ve düşün.** Bütün füzyon modelleri B hastanesindeki riski olduğundan düşük tahmin ediyor. B'nin dört farkından hangilerini girdiler gösterebilirdi, hangilerini gösteremezdi? Güncelleme neden farkın yalnızca bir bölümünü kapatıyor?

**Alıştırma I.** Güncellemeden sonra erken füzyonun B hastanesinin sonraki 700 hastasındaki Brier skoru (`pB_new[rest]`) kaçtır? Üç ondalık basamakla saklayın.

In [ ]:
veri = alistirma_tablosu("I")   # bu alıştırmanın promptundaki tablo (etkileşimli labın F bölümü)
cevap = None   # cevabınız
teach.check("Alıştırma I", cevap)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
cevap = round(float(brier_score_loss(yB[rest], pB_new[rest])), 3)
teach.check("Alıştırma I", cevap)

## 10. Taburculuk eşikleri ve net yarar

*Ders: [Taburculuk eşikleri ve net yarar](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-08/ders.html#thresholds-for-discharge-and-net-benefit)*

Erken füzyonun iki eşiği doğrulama başvurularında seçilir: Biri HEART 0-3 kadar hastayı eve gönderir, öteki duyarlılığı yüzde 98 düzeyinde tutar. İkisi de test başvurularında ve B hastanesinde, eve gönderilen hastalardaki MACE oranının Wilson aralıklarıyla birlikte, HEART 0-3'ün yanında sınanır. Ardından bir karar eğrisi yatırma kararı için modeli HEART ile karşılaştırır.

**Formüllerden koda.** Eşikler: `share_threshold(p, k)` ve `rule_out_threshold(y, p, 0.98)`. Wilson aralığı: `wilson(k, n)`. $\mathrm{NB}(t) = \mathrm{TP}/n - \mathrm{FP}/n \cdot t/(1 - t)$: `net_benefit(y, p, thresholds)`.

**Buradan mı başlıyorsunuz?** Aşağıdaki telafi hücresini bir kez çalıştırın. Hücre, 10. bölümün ihtiyaç duyduğu önceki bölümlerin kodunu, çıktılarını göstermeden çalıştırır.

In [ ]:
#@title Telafi: Jupyter Notebook'a buradan başlıyorsanız bu hücreyi bir kez çalıştırın
def _catch_up(upto, name="NB08_cok_modlu_yapay_zeka_ve_veri_fuzyonu.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-08/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Telafi tamamlandı: Önceki bölümlerin kodu çalıştırıldı.")


_catch_up(10)

In [ ]:
def share_threshold(p, k):
    """Tahmin edilen riski en düşük k hastayı eve gönderen eşik: En düşük k. risk."""
    return float(np.sort(p)[k - 1])


def rule_out_threshold(y, p, sensitivity):
    """Olaylar için duyarlılığı en az verilen düzeyde tutan en büyük eşik. Bu eşikte ya da altında kalan hastalar taburcu edilir. Eşik doğrulama kümesinde seçilir."""
    order = np.argsort(p)
    missed = np.cumsum(y[order])              # her eşikte ya da altında kalan olaylar
    ok = np.where(missed <= (1 - sensitivity) * y.sum())[0]
    return float(p[order][ok.max()]) if len(ok) else 0.0


def wilson(k, n, z=1.96):
    """Bir k/n oranının yüzde 95 Wilson aralığı."""
    if n == 0:
        return (0.0, 1.0)
    p = k / n
    c = (p + z * z / (2 * n)) / (1 + z * z / n)
    h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / (1 + z * z / n)
    return (c - h, c + h)


def net_benefit(y, p, thresholds):
    """Her eşiğin üstündeki hastaları tedavi etmenin net yararı: Hasta başına doğru pozitifler eksi eşiğin olabilirliğiyle ağırlıklandırılmış yanlış pozitifler."""
    n = len(y)
    return np.array([((p >= t) & (y == 1)).sum() / n - ((p >= t) & (y == 0)).sum() / n * t / (1 - t) for t in thresholds])


def discharge(home, yy, label):
    k, n = int(yy[home].sum()), int(home.sum())
    lo, hi = wilson(k, n)
    print(f"{label:34} eve: {n:4} ({home.mean():.3f})  MACE: {k:3}  ({k / n:.3f}; {lo:.3f}-{hi:.3f})  duyarlılık {1 - k / yy.sum():.3f}")


k_heart = int((heart[va] <= 3).sum())
t_share = share_threshold(P["early"][va], k_heart)
t_98 = rule_out_threshold(y[va], P["early"][va], 0.98)
print("eşikler:", round(t_share, 4), round(t_98, 4))
discharge(heart[te] <= 3, y[te], "HEART 0-3")
discharge(P["early"][te] <= t_share, y[te], "erken füzyon, aynı oran")
discharge(P["early"][te] <= t_98, y[te], "erken füzyon, yüzde 98")
print("B hastanesi, ilk 300 hastadan sonraki 700 hasta:")
discharge(B["heart"].to_numpy()[rest] <= 3, yB[rest], "HEART 0-3")
discharge(PB["early"][rest] <= t_share, yB[rest], "A'nın eşiği, güncellemeden önce")
discharge(pB_new[rest] <= t_share, yB[rest], "A'nın eşiği, güncellemeden sonra")

**Python notu.** `share_threshold` doğrulama risklerini sıralar ve en düşük k. riski alır; k, HEART skoru 0-3 olan doğrulama hastalarının sayısıdır. `rule_out_threshold` sıralanmış risklerde yukarı doğru ilerler, her birinde ya da altında kalan olayları `np.cumsum` ile sayar ve olayların yüzde 2'sinden fazlasının eve gönderilmeyeceği en büyük eşiği tutar. `discharge` bu hücrenin küçük bir fonksiyonudur ve bir kural için tek bir satır yazdırır: Kaç hastanın eve gittiği, oranları, bunlardaki MACE ve Wilson aralığı ile duyarlılık, yani bütün olayların hastanede tutulan oranı.

In [ ]:
ths = np.round(np.arange(0.01, 0.51, 0.01), 2)
curves = {"erken füzyon": net_benefit(y[te], P["early"][te], ths),
          "HEART ≥ 4": net_benefit(y[te], (heart[te] >= 4).astype(float), ths),
          "herkesi yatır": y[te].mean() - (1 - y[te].mean()) * ths / (1 - ths)}
fig, ax = plt.subplots(figsize=(6.4, 3.8))
for (name, nb), color in zip(curves.items(), ["#1F5F8B", "#B97813", "#8A8A8A"]):
    ax.plot(ths, nb, color=color, label=name)
ax.axhline(0, color="black", lw=0.8)
ax.set_ylim(-0.05, 0.2)
ax.set_xlabel("eşik olasılığı")
ax.set_ylabel("net yarar")
ax.legend()
plt.show()
for t in (0.05, 0.10, 0.20, 0.30):
    j = int(round(t * 100)) - 1
    print(f"t = {t:.2f}: " + "  ".join(f"{name} {nb[j]:.3f}" for name, nb in curves.items()))

**Python notu.** `np.arange(0.01, 0.51, 0.01)` yüzde 1 ile 50 arasındaki eşikleri verir. Evet ya da hayır kuralı olan HEART için `(heart[te] >= 4).astype(float)` dizisi olasılık yerine geçer: Yatırılan her hasta için 1, ötekiler için 0; böylece bu hastalar her eşikte yatırılır. Herkesi yatırmanın net yararı için model gerekmez: Olay oranından, olaysız hastaların oranı ile eşiğin olabilirliğinin çarpımı çıkarılır. Eşikler 0,01'den başlayıp 0,01 arttığı için `nb[j]` eğriyi `t` eşiğinde okur.

**Gözlemle ve düşün.** Taburculuk kararı için model HEART kadar iyidir; yüzde 20 ya da 30 eşikteki yatırma kararı için ise açıkça daha iyidir. Bunu bir kalp ekibine nasıl açıklardınız ve modele önce hangi kararda güvenirdiniz?

**Alıştırma J.** Erken füzyonun test başvurularında yüzde 10 eşik olasılığındaki net yararı kaçtır? Üç ondalık basamakla saklayın.

In [ ]:
veri = alistirma_tablosu("J")   # bu alıştırmanın promptundaki tablo (etkileşimli labın F bölümü)
cevap = None   # cevabınız
teach.check("Alıştırma J", cevap)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
cevap = round(float(net_benefit(y[te], P["early"][te], [0.10])[0]), 3)
teach.check("Alıştırma J", cevap)

**3. uygulama durağının Python yolu tamamlandı.** [Ders sayfasına dönün](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-08/ders.html#ai-pair-programming-part-8-reviewing-a-fusion-pipeline) ve okumaya «YZ asistanıyla çalışmak, 8. bölüm: Bir füzyon işlem hattını denetlemek» bölümünden devam edin. Bu Jupyter Notebook'un geri kalanı ek uygulamaları ve haftalık projeye yardımcı bölümleri içerir.

## Alanlara göre uygulama görevleri

| Alan | Görev |
|---|---|
| Acil kardiyoloji | Kılavuzlardaki HEART yolunu göğüs ağrısı ünitesinde kurun: HEART 0-3 ve iki troponinin de üst referans sınırında ya da altında olması. Bu yol, tek başına HEART 0-3'ün yanında kaç test hastasını eve gönderir ve kaç MACE kaçırır? |
| Girişimsel kardiyoloji | B hastanesindeki MACE'i türüne göre inceleyin. B'deki fazladan riskin ne kadarı anginadaki revaskülarizasyonlardan gelir ve yalnızca miyokard enfarktüsünü ve ölümü öngören bir model B'de ne gösterirdi? |
| Kalp yetersizliği | Kayıt sisteminde laboratuvardaki NT-proBNP'yi tek bir modelde ekokardiyograma ekleyin ve EKG'yi dışarıda bırakın. İki modaliteli bu model üçünün erken füzyonundan daha iyi midir ve farktan ne kadar emin olunabilir? |
| Klinik araştırma | B hastanesinin güncellemesini ilk 100, 200, 300, 500 ve 700 hastayla tekrarlayın ve her seferinde onlardan sonraki başvurularda sınayın. Sonraki hastalardaki kalibrasyon sabiti yerel hasta sayısıyla nasıl değişir? |
| Kadın ve kalp | Test başvurularında kadınlarda ve erkeklerde HEART ile erken füzyonu karşılaştırın: AUC, kalibrasyon sabiti ve aynı oran eşiğinde kaçırılan MACE. Kurallardan herhangi biri kadınlarda daha fazla olay kaçırıyor mu? |
| Kalite ve güvenlik | Erken füzyonun eve gönderdiği ve MACE görülen on test hastasını listeleyin. Tanıları, troponinleri, HEART skorları ve notları neydi? Ekip bunlardan hangilerini ve nasıl yakalayabilirdi? |
| Tıp bilişimi | Gerçek bir göğüs ağrısı ünitesi için bir veri çekme talebinin tanımını yazın: Sistemler, indeks zamanı, birimi ve zaman damgasıyla her girdi, MACE tanımı ve izlem kaynakları ile ilk modelden önceki denetimler. |

## Jupyter Notebook'tan sonra

[Etkileşimli labı](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-08/lab.html) açın ve *Kendini sına* sekmesindeki öz değerlendirmeyi yapın. Ardından *Yansıt ve dışa aktar* sekmesinde yansıtma yazınızı yazın ve öğrenme günlüğünü dışa aktarın. Bu haftanın haftalık projesi 7, 9 ve 10. bölümleri genişletir: Modeller aralıklarıyla HEART ile karşılaştırılır, kalibrasyonları iki hastanede değerlendirilir, iki taburculuk eşiği sınanır ve bir kalp ekibi için karar eğrisi çizilir.

<div align="center">

---

**Prof. Dr. Utku Köse**<br>
Profesör, Bilgisayar Mühendisliği Bölümü, Süleyman Demirel Üniversitesi, Isparta, Türkiye<br>
Kurucu Müdür, Yapay Zekâ Uygulama ve Araştırma Merkezi (YAZEM), Süleyman Demirel Üniversitesi<br>
Bilgisayar Bilimleri Anabilim Dalı Başkanı, Bilgisayar Mühendisliği Bölümü, Süleyman Demirel Üniversitesi<br>
Diğer kurumlar: University of North Dakota (ABD), Universidad Panamericana (Meksiko, Meksika), Vel Tech University (Chennai, Hindistan)<br>
IEEE Kıdemli Üyesi, ACM Profesyonel Üyesi

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) · [utkukose.com](https://www.utkukose.com) · [github.com/utkukose](https://github.com/utkukose)

*Python ile Yapay Zekâ Eğitimi: Kardiyologlar için Uygulamalı Eğitim Programı, Türk Kardiyoloji Derneği. İlk sürüm: Kasım 2026. İçerik CC BY 4.0, kod MIT.*

</div>